# Kaggriculture 401st (silver): `rank95_front97`

This notebook publishes the agent I submitted last (submission `56720329`, 2026-10-01 07:34 JST).
Its hosted score was 1057.9 right after submission and settled at **2185.6, rank 401 (silver medal)**.

## What the agent is

A **rule-based route-replay agent distilled from public replays**. No neural network and no reinforcement learning.

- **Base route**: the 719 actions that the rank-95 team played in public episode `115755103`
  (stored in `main.py` as a zlib/base85-compressed blob, together with the rank-97 sell schedule).
- **Corrections** (the `Chassis` class): hand alignment, weed repair, sell-lead handling, budget guard, warehouse room guard,
  sell-quantity clamping, dead-stock handling and terminal liquidation.
- **front97**: a public development replay of the rank-97 team gives a predicted sell schedule. Only for goods that the agent already
  holds and plans to sell on the next turn, the market sale is moved one turn earlier. No private observation of the opponent is used.
- **Selection**: 645 public games of the top-100 teams were split into development and validation sets. 463 action routes from the
  development games, 35 candidates and 4 existing agents were played off in round robins with the official engine
  (`kaggle-environments` 1.32.7, both seats). The final local result was 436 wins and 268 losses over 704 games.

## Honest limits

- The local win rate did **not** predict the hosted rating: the same agent showed 1057.9 on the leaderboard right after submission.
  I could not determine why it later rose to 2185.6. Treat local win rates as weak evidence.
- The routes come from public replays. No competitor source code, hidden observation or full replay is included.
- This is a notebook-style republication of the submitted file, not a new method.
- The notebook checks that `main.py` is byte-identical to the submitted file (SHA-256 below) and builds `submission.tar.gz`.
  It does not play games: the Kaggle notebook image has no Kaggriculture environment. The official-engine matches were run
  outside this notebook with `kaggle-environments` 1.32.7.

## Identity

- `main.py` SHA-256: `4b4a4c4b63957b8c095f789a045505c7a238999d315d7f518237e5a7b9f34db6` (77,970 bytes, CRLF line endings)
- Submitted archive SHA-256: `8324ece0db04da2b7948e70320b167bcdf44497ce3189669e3b4ef4e03a1e2f1`
  (the rebuilt archive differs in tar/gzip metadata such as timestamps; its `main.py` member is byte-identical)

## Attribution and license

The chassis derives from public Apache-2.0 notebooks. The header of `main.py` keeps the credits for
thomastschinkel, yhay81, destbreso, aurax7, tetsutani, prvsiyan, Dmitrii Gluzdov, Rayk Kretzschmar and Ahmed Berat Ozer.
This notebook is released under **Apache-2.0**, the same license as its upstream sources.


In [ ]:
%%writefile main.py
# EXP260 combined; V41 remains the frozen primary control.
# Earlier reservation activation adapted from aurax7 Reactive V5; upstream notices retained.
# EXP257: V39 base with funded atomic opening.
# Opening assignment adapted from Rayk Kretzschmar, Rank Your Agent.
# Existing upstream attribution retained below; original funding guard by Ahmed Berat Ozer.
# EXP-173 isolate opening market sequence inspired by yhay81/shop-router-0911-simple (Apache-2.0).
# Kaggriculture EXP-167 candidate. Not submitted automatically.
# Attribution: thomastschinkel, yhay81, destbreso, aurax7, tetsutani,
# prvsiyan and Dmitrii Gluzdov. Apache-2.0 derivations; notices retained below.
# Kaggriculture v31 / EXP-157, Ahmed Berat Ozer, September 9 2026.
# Selected mechanism: crop_public_order. New independent confirmation is required.
# Public V221B/V224C production/timing lineage: prvsiyan, Apache-2.0.
# Original economics and integration; retained upstream licenses follow.
# Kaggriculture v28 / EXP-154, Ahmed Berat Ozer, September 9 2026.
# Changes: aurax7 day-end storage guard; Dmitrii Gluzdov physical terminal rescue
# adapted to v27, with 64 deterministic simulations. Apache-2.0.
# New action tapes and ordered shop-pair map: yhay81/shop-router-0909, Apache-2.0.
# Kaggriculture v25, EXP-149: Shop0908 production, sale lead, terminal cargo rescue.
# Runtime chassis: Apache-2.0; thomastschinkel, yhay81, tetsutani.
# Routing and public action data: yhay81/shop-router-0908, frozen September 8, 2026.
# 
#                                  Apache License
#                            Version 2.0, January 2004
#                         http://www.apache.org/licenses/
# 
#    TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION
# 
#    1. Definitions.
# 
#       "License" shall mean the terms and conditions for use, reproduction,
#       and distribution as defined by Sections 1 through 9 of this document.
# 
#       "Licensor" shall mean the copyright owner or entity authorized by
#       the copyright owner that is granting the License.
# 
#       "Legal Entity" shall mean the union of the acting entity and all
#       other entities that control, are controlled by, or are under common
#       control with that entity. For the purposes of this definition,
#       "control" means (i) the power, direct or indirect, to cause the
#       direction or management of such entity, whether by contract or
#       otherwise, or (ii) ownership of fifty percent (50%) or more of the
#       outstanding shares, or (iii) beneficial ownership of such entity.
# 
#       "You" (or "Your") shall mean an individual or Legal Entity
#       exercising permissions granted by this License.
# 
#       "Source" form shall mean the preferred form for making modifications,
#       including but not limited to software source code, documentation
#       source, and configuration files.
# 
#       "Object" form shall mean any form resulting from mechanical
#       transformation or translation of a Source form, including but
#       not limited to compiled object code, generated documentation,
#       and conversions to other media types.
# 
#       "Work" shall mean the work of authorship, whether in Source or
#       Object form, made available under the License, as indicated by a
#       copyright notice that is included in or attached to the work
#       (an example is provided in the Appendix below).
# 
#       "Derivative Works" shall mean any work, whether in Source or Object
#       form, that is based on (or derived from) the Work and for which the
#       editorial revisions, annotations, elaborations, or other modifications
#       represent, as a whole, an original work of authorship. For the purposes
#       of this License, Derivative Works shall not include works that remain
#       separable from, or merely link (or bind by name) to the interfaces of,
#       the Work and Derivative Works thereof.
# 
#       "Contribution" shall mean any work of authorship, including
#       the original version of the Work and any modifications or additions
#       to that Work or Derivative Works thereof, that is intentionally
#       submitted to Licensor for inclusion in the Work by the copyright owner
#       or by an individual or Legal Entity authorized to submit on behalf of
#       the copyright owner. For the purposes of this definition, "submitted"
#       means any form of electronic, verbal, or written communication sent
#       to the Licensor or its representatives, including but not limited to
#       communication on electronic mailing lists, source code control systems,
#       and issue tracking systems that are managed by, or on behalf of, the
#       Licensor for the purpose of discussing and improving the Work, but
#       excluding communication that is conspicuously marked or otherwise
#       designated in writing by the copyright owner as "Not a Contribution."
# 
#       "Contributor" shall mean Licensor and any individual or Legal Entity
#       on behalf of whom a Contribution has been received by Licensor and
#       subsequently incorporated within the Work.
# 
#    2. Grant of Copyright License. Subject to the terms and conditions of
#       this License, each Contributor hereby grants to You a perpetual,
#       worldwide, non-exclusive, no-charge, royalty-free, irrevocable
#       copyright license to reproduce, prepare Derivative Works of,
#       publicly display, publicly perform, sublicense, and distribute the
#       Work and such Derivative Works in Source or Object form.
# 
#    3. Grant of Patent License. Subject to the terms and conditions of
#       this License, each Contributor hereby grants to You a perpetual,
#       worldwide, non-exclusive, no-charge, royalty-free, irrevocable
#       (except as stated in this section) patent license to make, have made,
#       use, offer to sell, sell, import, and otherwise transfer the Work,
#       where such license applies only to those patent claims licensable
#       by such Contributor that are necessarily infringed by their
#       Contribution(s) alone or by combination of their Contribution(s)
#       with the Work to which such Contribution(s) was submitted. If You
#       institute patent litigation against any entity (including a
#       cross-claim or counterclaim in a lawsuit) alleging that the Work
#       or a Contribution incorporated within the Work constitutes direct
#       or contributory patent infringement, then any patent licenses
#       granted to You under this License for that Work shall terminate
#       as of the date such litigation is filed.
# 
#    4. Redistribution. You may reproduce and distribute copies of the
#       Work or Derivative Works thereof in any medium, with or without
#       modifications, and in Source or Object form, provided that You
#       meet the following conditions:
# 
#       (a) You must give any other recipients of the Work or
#           Derivative Works a copy of this License; and
# 
#       (b) You must cause any modified files to carry prominent notices
#           stating that You changed the files; and
# 
#       (c) You must retain, in the Source form of any Derivative Works
#           that You distribute, all copyright, patent, trademark, and
#           attribution notices from the Source form of the Work,
#           excluding those notices that do not pertain to any part of
#           the Derivative Works; and
# 
#       (d) If the Work includes a "NOTICE" text file as part of its
#           distribution, then any Derivative Works that You distribute must
#           include a readable copy of the attribution notices contained
#           within such NOTICE file, excluding those notices that do not
#           pertain to any part of the Derivative Works, in at least one
#           of the following places: within a NOTICE text file distributed
#           as part of the Derivative Works; within the Source form or
#           documentation, if provided along with the Derivative Works; or,
#           within a display generated by the Derivative Works, if and
#           wherever such third-party notices normally appear. The contents
#           of the NOTICE file are for informational purposes only and
#           do not modify the License. You may add Your own attribution
#           notices within Derivative Works that You distribute, alongside
#           or as an addendum to the NOTICE text from the Work, provided
#           that such additional attribution notices cannot be construed
#           as modifying the License.
# 
#       You may add Your own copyright statement to Your modifications and
#       may provide additional or different license terms and conditions
#       for use, reproduction, or distribution of Your modifications, or
#       for any such Derivative Works as a whole, provided Your use,
#       reproduction, and distribution of the Work otherwise complies with
#       the conditions stated in this License.
# 
#    5. Submission of Contributions. Unless You explicitly state otherwise,
#       any Contribution intentionally submitted for inclusion in the Work
#       by You to the Licensor shall be under the terms and conditions of
#       this License, without any additional terms or conditions.
#       Notwithstanding the above, nothing herein shall supersede or modify
#       the terms of any separate license agreement you may have executed
#       with Licensor regarding such Contributions.
# 
#    6. Trademarks. This License does not grant permission to use the trade
#       names, trademarks, service marks, or product names of the Licensor,
#       except as required for reasonable and customary use in describing the
#       origin of the Work and reproducing the content of the NOTICE file.
# 
#    7. Disclaimer of Warranty. Unless required by applicable law or
#       agreed to in writing, Licensor provides the Work (and each
#       Contributor provides its Contributions) on an "AS IS" BASIS,
#       WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or
#       implied, including, without limitation, any warranties or conditions
#       of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A
#       PARTICULAR PURPOSE. You are solely responsible for determining the
#       appropriateness of using or redistributing the Work and assume any
#       risks associated with Your exercise of permissions under this License.
# 
#    8. Limitation of Liability. In no event and under no legal theory,
#       whether in tort (including negligence), contract, or otherwise,
#       unless required by applicable law (such as deliberate and grossly
#       negligent acts) or agreed to in writing, shall any Contributor be
#       liable to You for damages, including any direct, indirect, special,
#       incidental, or consequential damages of any character arising as a
#       result of this License or out of the use or inability to use the
#       Work (including but not limited to damages for loss of goodwill,
#       work stoppage, computer failure or malfunction, or any and all
#       other commercial damages or losses), even if such Contributor
#       has been advised of the possibility of such damages.
# 
#    9. Accepting Warranty or Additional Liability. While redistributing
#       the Work or Derivative Works thereof, You may choose to offer,
#       and charge a fee for, acceptance of support, warranty, indemnity,
#       or other liability obligations and/or rights consistent with this
#       License. However, in accepting such obligations, You may act only
#       on Your own behalf and on Your sole responsibility, not on behalf
#       of any other Contributor, and only if You agree to indemnify,
#       defend, and hold each Contributor harmless for any liability
#       incurred by, or claims asserted against, such Contributor by reason
#       of your accepting any such warranty or additional liability.
# 
#    END OF TERMS AND CONDITIONS
# 
#    APPENDIX: How to apply the Apache License to your work.
# 
#       To apply the Apache License to your work, attach the following
#       boilerplate notice, with the fields enclosed by brackets "[]"
#       replaced with your own identifying information. (Don't include
#       the brackets!)  The text should be enclosed in the appropriate
#       comment syntax for the file format. We also recommend that a
#       file or class name and description of purpose be included on the
#       same "printed page" as the copyright notice for easier
#       identification within third-party archives.
# 
#    Copyright [yyyy] [name of copyright owner]
# 
#    Licensed under the Apache License, Version 2.0 (the "License");
#    you may not use this file except in compliance with the License.
#    You may obtain a copy of the License at
# 
#        http://www.apache.org/licenses/LICENSE-2.0
# 
#    Unless required by applicable law or agreed to in writing, software
#    distributed under the License is distributed on an "AS IS" BASIS,
#    WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
#    See the License for the specific language governing permissions and
#    limitations under the License.
"""Kaggriculture route-replay chassis (pure Python, stdlib only).

A *route* is a pre-computed tape of 719 Kaggle-format actions
``{"farmer": [op, ...], "hands": [[op, ...], ...], "market": [[order, item, qty], ...]}``.
The chassis replays the tape chosen by a caller-supplied ``router`` and wraps it
in small reactive layers (each independently switchable via ``settings``):

    hand_align            pad/truncate hands to the real hand count      (fieldbook_logic)
    weed_repair           DIG a weed that blocks PLANT/BUILD, replay      (tetsutani + task spec)
    sell_lead             sell next step's lots one step early            (fieldbook _lead_sale)
    front_run             sell before the opponent's scheduled SELL       (hook; opponent_plan)
    budget_guard          fund each 72-step block's purchases             (six_day_budget_guard.hpp)
    room_guard            keep shed <= 99 at hour 23                      (tetsutani)
    clamp_sells           trim SELL orders to the projected shed          (tetsutani)
    dead_stock            sell stock the route will never sell            (tetsutani)
    terminal_liquidation  step >= 718: sell the whole projected shed      (fieldbook _terminal_sale)

Engine facts (verified against kaggle_environments 1.32.7, env_1_32_7.py):
  observation["farms"][p] = {"money", "tiles"[y][x], "farmer"[x,y], "hands"[[x,y]..],
                             "unlocked_quadrants", "hires_today"}
  tiles: None (empty) | "LOCKED" | {"kind": WEED|COOP|PASTURE|PLANT, "crop"/"animal", ...}
  observation["private"] = {"shed": {item: n}, "seeds": {crop: n}, "inventories": [{}...]}
  observation["market"] = {"inventory": {...}, "prices": {...}}
  observation["town"] = {"unlocked_shops": [...]}
  Agents act on steps 0..718 (interpreter marks DONE once step >= episodeSteps-2).
"""
from __future__ import annotations

import copy

PRODUCTS = ("WHEAT", "CARROT", "TOMATO", "STRAWBERRY", "MELON", "EGG", "MILK", "WOOL", "FERTILIZER")
SEED_PRICE = {"WHEAT": 10, "CARROT": 20, "TOMATO": 50, "STRAWBERRY": 100, "MELON": 80}
ANIMAL_COST = {"GOOSE": 300, "COW": 400, "SHEEP": 500}
ANIMAL_STRUCTURE = {"GOOSE": "COOP", "COW": "PASTURE", "SHEEP": "PASTURE"}
LAND_PRICES = (1000, 2000, 4000)
MOVES = {"NORTH": (0, -1), "SOUTH": (0, 1), "EAST": (1, 0), "WEST": (-1, 0)}
FRONT_RUN_ITEMS = ("MILK", "WOOL", "STRAWBERRY", "MELON")
LAST_ACT_STEP = 718
PASS_ACTION = {"farmer": ["PASS"], "hands": [], "market": []}

DEFAULT_SETTINGS = {
    "hand_align": True,
    "weed_repair": True,
    "sell_lead": True,
    "front_run": True,
    "budget_guard": True,
    "room_guard": True,
    "clamp_sells": True,
    "dead_stock": True,
    "terminal_liquidation": True,
    # tunables
    "block_turns": 72,
    "shed_capacity": 100,
    "board_size": 10,
    "max_orders": 10,
    "turns_per_day": 24,
    "min_sell_price": 2,
}


# --------------------------------------------------------------------------- helpers
def _get(value, key, default=None):
    """Field access that works for dicts and Kaggle Struct/attribute objects."""
    if isinstance(value, dict):
        return value.get(key, default)
    getter = getattr(value, "get", None)
    if callable(getter):
        return getter(key, default)
    return getattr(value, key, default)


def _int(value, default=0):
    try:
        return int(value)
    except (TypeError, ValueError):
        return default


def _fib(n):
    a, b = 1, 1
    for _ in range(n):
        a, b = b, a + b
    return a


def _step_of(observation):
    raw = _get(observation, "step")
    if raw is not None:
        return _int(raw)
    return _int(_get(observation, "day", 0)) * 24 + _int(_get(observation, "hour", 0))


def _shed_adjacent(pos, board):
    if not isinstance(pos, (list, tuple)) or len(pos) < 2:
        return False
    half = board // 2
    return pos[0] in (half - 1, half) and pos[1] in (half - 1, half)


def _tile_at(tiles, pos):
    try:
        x, y = int(pos[0]), int(pos[1])
        return tiles[y][x]
    except (TypeError, ValueError, IndexError):
        return "LOCKED"


def _is_noop(act, tile, inv, seeds, pos, board):
    """True when the engine will certainly ignore ``act`` (mirrors _apply_unit_action)."""
    if not act:
        return True
    op = act[0]
    x, y = pos[0], pos[1]
    if op in MOVES:
        dx, dy = MOVES[op]
        return not (0 <= x + dx < board and 0 <= y + dy < board)
    if op == "PASS":
        return True
    adjacent = _shed_adjacent(pos, board)
    if op == "DROP":
        return (not adjacent) or (not inv)
    if op == "PICKUP":
        return not adjacent
    if op == "PLACE":
        item = act[1] if len(act) > 1 else None
        if item in ANIMAL_STRUCTURE and isinstance(tile, dict) \
                and _get(tile, "kind") == ANIMAL_STRUCTURE[item] and _get(tile, "animal") is None:
            return _int(_get(inv, item, 0)) <= 0
        return (not adjacent) or _int(_get(inv, item, 0)) <= 0
    if tile == "LOCKED":
        return True
    is_dict = isinstance(tile, dict)
    kind = _get(tile, "kind") if is_dict else None
    animal = is_dict and _get(tile, "animal") is not None
    if op == "PLANT":
        return tile is not None or _int(_get(seeds, act[1] if len(act) > 1 else None, 0)) <= 0
    if op == "WATER":
        return kind != "PLANT" or bool(_get(tile, "watered_today"))
    if op == "HARVEST":
        return (not is_dict) or _int(_get(tile, "yield_units", 0)) <= 0
    if op == "FERTILIZE":
        return kind != "PLANT" or _int(_get(inv, "FERTILIZER", 0)) <= 0
    if op == "DIG":
        return tile is None or animal
    if op in ("BUILD_COOP", "BUILD_PASTURE"):
        return tile is not None
    if op == "FEED":
        return (not animal) or bool(_get(tile, "fed_today")) or _int(_get(inv, "WHEAT", 0)) <= 0
    if op == "COLLECT_FERTILIZER":
        return (not animal) or (not _get(tile, "fertilizer_available"))
    if op == "CARE":
        return (not animal) or bool(_get(tile, "cared_today"))
    return True


class _View:
    """Cheap per-step snapshot of everything the layers read from the observation."""

    def __init__(self, observation, player, cfg):
        farms = list(_get(observation, "farms", []) or [])
        self.farm = farms[player] if player < len(farms) else {}
        self.rival = farms[1 - player] if len(farms) >= 2 and 1 - player < len(farms) else {}
        private = _get(observation, "private", {}) or {}
        self.shed = {k: max(0, _int(v)) for k, v in dict(_get(private, "shed", {}) or {}).items()}
        self.seeds = dict(_get(private, "seeds", {}) or {})
        self.invs = [dict(i or {}) for i in (_get(private, "inventories", []) or [])]
        market = _get(observation, "market", {}) or {}
        self.prices = {k: _int(v) for k, v in dict(_get(market, "prices", {}) or {}).items()}
        self.money = float(_get(self.farm, "money", 0.0) or 0.0)
        self.tiles = _get(self.farm, "tiles", []) or []
        self.board = len(self.tiles) or cfg["board_size"]
        self.positions = [_get(self.farm, "farmer", None)] + [list(p) for p in (_get(self.farm, "hands", []) or [])]
        self.hires_today = _int(_get(self.farm, "hires_today", 0))
        self.quadrants = len(list(_get(self.farm, "unlocked_quadrants", []) or []))

    def inv(self, idx):
        return self.invs[idx] if idx < len(self.invs) else {}

    def in_hands(self, item):
        return sum(max(0, _int(_get(inv, item, 0))) for inv in self.invs)


# --------------------------------------------------------------------------- chassis
class Chassis:
    """Replays ``routes[router(...)]`` with reactive safety/market layers.

    routes         : {route_id: list of >= 719 Kaggle action dicts}
    router         : callable(observation, step, state_dict) -> route_id, called every
                     step; ``state_dict`` is per-player and persists across the game.
    settings       : overrides for DEFAULT_SETTINGS (layer switches + tunables)
    opponent_plan  : optional list of the opponent's expected actions (front_run hook)
    """

    def __init__(self, routes, router=None, settings=None, opponent_plan=None):
        self.routes = {rid: list(tape) for rid, tape in routes.items()}
        self.router = router or (lambda observation, step, state: next(iter(self.routes)))
        self.cfg = dict(DEFAULT_SETTINGS)
        self.cfg.update(settings or {})
        self.opponent_plan = opponent_plan
        self.players = {}
        self.diagnostics = {"layer_fallbacks": 0, "entry_fallbacks": 0}
        self._future_sells = {}   # route id -> {item: [remaining planned SELL qty from step t]}

    # ---- state -----------------------------------------------------------------
    def _state(self, player, step):
        st = self.players.get(player)
        if st is None or step == 0 or step <= st["last_step"]:
            st = {"last_step": -1, "route": None, "router_state": {},
                  "pending": {}, "sell_state": {"due_step": -1, "suppress": {}}}
            self.players[player] = st
        st["last_step"] = step
        return st

    def _route_action(self, route, step):
        tape = self.routes[route]
        if 0 <= step < len(tape) and isinstance(tape[step], dict):
            return copy.deepcopy(tape[step])
        return copy.deepcopy(PASS_ACTION)

    def future_sells(self, route, item, step):
        """Planned SELL quantity of ``item`` in route steps >= ``step`` (suffix sums)."""
        table = self._future_sells.get(route)
        if table is None:
            tape = self.routes[route]
            n = len(tape)
            table = {p: [0] * (n + 1) for p in PRODUCTS}
            for t in range(n - 1, -1, -1):
                for p in PRODUCTS:
                    table[p][t] = table[p][t + 1]
                for o in (tape[t].get("market") or []) if isinstance(tape[t], dict) else []:
                    if o and o[0] == "SELL" and len(o) >= 3 and o[1] in table:
                        table[o[1]][t] += max(0, _int(o[2]))
            self._future_sells[route] = table
        col = table.get(item)
        return col[step] if col and 0 <= step < len(col) else 0

    # ---- main entry -----------------------------------------------------------
    def act(self, observation, configuration=None):
        if len(_get(observation, "farms", []) or []) < 2:
            raise ValueError("incomplete observation")  # factory falls back to tape
        step = _step_of(observation)
        player = _int(_get(observation, "player", 0))
        st = self._state(player, step)
        cfg = self.cfg
        view = _View(observation, player, cfg)

        route = self.router(observation, step, st["router_state"])
        if route not in self.routes:
            route = st["route"] if st["route"] in self.routes else next(iter(self.routes))
        st["route"] = route
        action = self._route_action(route, step)
        raw = copy.deepcopy(action)
        try:
            if cfg["hand_align"]:
                self._hand_align(action, view)
            if cfg["weed_repair"]:
                self._weed_repair(action, view, st, route, step)
            if cfg["sell_lead"] or cfg["front_run"]:
                self._apply_suppression(action, st["sell_state"], step)
            projected = self._projected_shed(action, view)
            lead_available = dict(projected)
            next_sup = {"due_step": -1, "suppress": {}, "r36_debts": st["sell_state"].get("r36_debts", {})}
            if cfg["sell_lead"]:
                self._sell_lead(action, view, lead_available, route, step, next_sup)
            if cfg["front_run"] and self.opponent_plan:
                self._front_run(action, view, lead_available, route, step, next_sup)
            st["sell_state"] = next_sup
            if cfg["budget_guard"]:
                self._budget_guard(action, view, route, step)
            if cfg["room_guard"]:
                self._room_guard(action, view, route, step)
            if cfg["clamp_sells"]:
                self._clamp_sells(action, projected)
            if cfg["dead_stock"]:
                self._dead_stock(action, view, projected, route, step)
            if cfg["terminal_liquidation"]:
                self._terminal_liquidation(action, projected, step)
            action["market"] = action["market"][: cfg["max_orders"]]
            return action
        except Exception:
            self.diagnostics["layer_fallbacks"] += 1
            return raw

    # ---- layer: hand_align ----------------------------------------------------
    def _hand_align(self, action, view):
        """Pad with PASS / truncate the tape's hand list to the real number of hands
        (fieldbook_logic.act). Extra hands would be ignored by the engine anyway;
        missing ones just idle, so alignment only tidies the action."""
        expected = max(0, len(view.positions) - 1)
        hands = list(action.get("hands") or [])
        hands.extend([["PASS"] for _ in range(max(0, expected - len(hands)))])
        action["hands"] = hands[:expected]

    # ---- layer: weed_repair ---------------------------------------------------
    def _weed_repair(self, action, view, st, route, step):
        """If a PLANT/BUILD_* target tile is a WEED, DIG now and queue the intended
        action for that unit; the queue replays on a later step when the unit still
        stands there and its tape action would be a no-op (the displaced no-op is
        queued behind it, so PLANT -> WATER chains survive). A PLANT is only replayed
        when the unit's next tape action is not a move, so the mandatory same-day
        WATER can follow; otherwise the seed is kept. A no-op turn spent on a weed
        is also converted to DIG (tetsutani weed_dig)."""
        units = [action.get("farmer") or ["PASS"]] + list(action.get("hands") or [])
        pending = st["pending"]
        tape = self.routes[route]
        nxt = tape[step + 1] if step + 1 < len(tape) and isinstance(tape[step + 1], dict) else {}
        next_units = [nxt.get("farmer") or ["PASS"]] + list(nxt.get("hands") or [])
        for i in range(min(len(units), len(view.positions))):
            pos = view.positions[i]
            if not isinstance(pos, (list, tuple)):
                continue
            pos = (int(pos[0]), int(pos[1]))
            tile = _tile_at(view.tiles, pos)
            act = list(units[i])
            queue = pending.get(i)
            if queue and queue[0][0] != pos:
                pending.pop(i, None)
                queue = None
            is_weed = isinstance(tile, dict) and _get(tile, "kind") == "WEED"
            noop = _is_noop(act, tile, view.inv(i), view.seeds, pos, view.board)
            next_op = next_units[i][0] if i < len(next_units) and next_units[i] else "PASS"
            if act and act[0] in ("PLANT", "BUILD_COOP", "BUILD_PASTURE") and is_weed:
                pending.setdefault(i, []).append((pos, act))
                act = ["DIG"]
            elif queue and noop:
                _, replay = queue[0]
                if replay[0] == "PLANT" and next_op in MOVES:
                    pending.pop(i, None)          # WATER could never follow: keep the seed
                else:
                    queue.pop(0)
                    if act and act[0] != "PASS" and act[0] not in MOVES:
                        queue.append((pos, act))
                    act = replay
                    if not queue:
                        pending.pop(i, None)
            elif is_weed and noop:
                act = ["DIG"]
            units[i] = act
        action["farmer"] = units[0]
        action["hands"] = units[1:]

    # ---- projected shed -------------------------------------------------------
    def _projected_shed(self, action, view):
        """Shed contents after this step's unit actions but before the market runs:
        PICKUP removes, DROP/PLACE(non-animal) near the shed adds up to capacity
        (fieldbook _projected_shed / tetsutani projected shed)."""
        cap = self.cfg["shed_capacity"]
        proj = {p: view.shed.get(p, 0) for p in PRODUCTS}
        for k, v in view.shed.items():
            proj.setdefault(k, v)
        total = sum(proj.values())
        units = [action.get("farmer") or ["PASS"]] + list(action.get("hands") or [])
        for i in range(min(len(units), len(view.positions))):
            if not _shed_adjacent(view.positions[i], view.board):
                continue
            act = units[i]
            op = act[0] if act else "PASS"
            inv = view.inv(i)
            if op == "PICKUP" and len(act) >= 2 and act[1] in proj:
                qty = min(proj[act[1]], max(0, _int(act[2]) if len(act) >= 3 else 1))
                proj[act[1]] -= qty
                total -= qty
            elif op == "DROP":
                for item, held in inv.items():
                    take = min(max(0, _int(held)), max(0, cap - total))
                    if take > 0:
                        proj[item] = proj.get(item, 0) + take
                        total += take
            elif op == "PLACE" and len(act) >= 2 and act[1] not in ANIMAL_STRUCTURE:
                item = act[1]
                take = min(max(0, _int(act[2]) if len(act) >= 3 else 1),
                           max(0, _int(_get(inv, item, 0))), max(0, cap - total))
                if take > 0:
                    proj[item] = proj.get(item, 0) + take
                    total += take
        return proj

    # ---- layer: sell_lead / front_run suppression ------------------------------
    @staticmethod
    def _apply_suppression(action, sell_state, step):
        """Remove from this step's SELLs the quantities already sold a step early."""
        if sell_state.get("due_step") != step:
            return
        remaining = dict(sell_state.get("suppress", {}))
        kept = []
        for order in action.get("market") or []:
            order = list(order)
            if order and order[0] == "SELL" and len(order) >= 3 and remaining.get(order[1], 0) > 0:
                removed = min(max(0, _int(order[2])), remaining[order[1]])
                order[2] = _int(order[2]) - removed
                remaining[order[1]] -= removed
                # A zero-quantity order keeps later market race slots intact.
            kept.append(order)
        action["market"] = kept

    @staticmethod
    def _add_sell(action, item, qty, max_orders, merge=True):
        market = action.setdefault("market", [])
        if merge:
            for order in market:
                if order and order[0] == "SELL" and order[1] == item:
                    order[2] = _int(order[2]) + qty
                    return True
        if len(market) >= max_orders:
            return False
        market.append(["SELL", item, qty])
        return True

    def _sell_lead(self, action, view, projected, route, step, next_sup):
        """fieldbook _lead_sale: when step % 4 != 0 (no town consumption between the
        two steps) sell the lots the tape plans to SELL next step now, for products
        other than WHEAT/FERTILIZER we already hold, and suppress them next step.
        Skipped at the last step, at shop-unlock boundaries and if a SELL for that
        product is already queued this step."""
        cfg = self.cfg
        nxt = step + 1
        unlock_period = 3 * cfg["turns_per_day"]
        if nxt > LAST_ACT_STEP or nxt % unlock_period == 0 or step % 4 == 0:
            return
        tape = self.routes[route]
        future = tape[nxt] if nxt < len(tape) and isinstance(tape[nxt], dict) else {}
        planned = {}
        for o in future.get("market") or []:
            if o and o[0] == "SELL" and len(o) >= 3 and o[1] in PRODUCTS:
                planned[o[1]] = planned.get(o[1], 0) + max(0, _int(o[2]))
        already = {o[1] for o in action.get("market") or [] if o and o[0] == "SELL" and len(o) > 1}
        for item in PRODUCTS:
            if item in ("WHEAT", "FERTILIZER") or planned.get(item, 0) <= 0 or item in already:
                continue
            qty = min(projected.get(item, 0), planned[item])
            if qty <= 0 or view.prices.get(item, 0) < cfg["min_sell_price"]:
                continue
            if not self._add_sell(action, item, qty, cfg["max_orders"], merge=False):
                break
            projected[item] -= qty
            next_sup["suppress"][item] = next_sup["suppress"].get(item, 0) + qty
        if next_sup["suppress"]:
            next_sup["due_step"] = nxt

    def _front_run(self, action, view, projected, route, step, next_sup):
        """Hook: if ``opponent_plan`` (their expected tape) schedules a SELL of
        MILK/WOOL/STRAWBERRY/MELON next step, sell what we hold of it now (before
        their supply depresses the price) and suppress our own SELL of that quantity
        next step. Bounded by our own remaining planned sales so it never dumps."""
        cfg = self.cfg
        nxt = step + 1
        plan = self.opponent_plan
        if nxt > LAST_ACT_STEP or nxt >= len(plan) or not isinstance(plan[nxt], dict):
            return
        already = {o[1] for o in action.get("market") or [] if o and o[0] == "SELL" and len(o) > 1}
        for o in plan[nxt].get("market") or []:
            if not (o and o[0] == "SELL" and len(o) >= 3 and o[1] in FRONT_RUN_ITEMS):
                continue
            item = o[1]
            if item in already or view.prices.get(item, 0) < cfg["min_sell_price"]:
                continue
            own_next = sum(max(0, _int(x[2])) for x in self.routes[route][nxt].get("market", [])
                           if len(x) >= 3 and x[0] == "SELL" and x[1] == item)
            qty = min(projected.get(item, 0), max(0, _int(o[2])), own_next)
            if qty <= 0:
                continue
            if not self._add_sell(action, item, qty, cfg["max_orders"], merge=False):
                break
            projected[item] -= qty
            already.add(item)
            next_sup["suppress"][item] = next_sup["suppress"].get(item, 0) + qty
        if next_sup["suppress"]:
            next_sup["due_step"] = nxt

    # ---- layer: budget_guard --------------------------------------------------
    def _block_requirements(self, view, route, start, end):
        """Planned purchase cost and item reserves for tape steps [start, end)
        (six_day_budget_guard.hpp calculate_six_day_requirements)."""
        tape = self.routes[route]
        budget = 0.0
        seed_bal, item_bal = {}, {}
        seed_need, item_need = {}, {}
        hires_by_day = {}
        quadrants = view.quadrants
        for t in range(start, min(end, len(tape))):
            a = tape[t] if isinstance(tape[t], dict) else {}
            for u in [a.get("farmer") or ["PASS"]] + list(a.get("hands") or []):
                if not u:
                    continue
                op = u[0]
                arg = u[1] if len(u) > 1 else None
                qty = max(1, _int(u[2]) if len(u) > 2 else 1)
                if op == "PLANT" and arg in SEED_PRICE:
                    seed_bal[arg] = seed_bal.get(arg, 0) - 1
                    seed_need[arg] = max(seed_need.get(arg, 0), -seed_bal[arg])
                elif op == "FEED":
                    item_bal["WHEAT"] = item_bal.get("WHEAT", 0) - 1
                    item_need["WHEAT"] = max(item_need.get("WHEAT", 0), -item_bal["WHEAT"])
                elif op == "FERTILIZE":
                    item_bal["FERTILIZER"] = item_bal.get("FERTILIZER", 0) - 1
                    item_need["FERTILIZER"] = max(item_need.get("FERTILIZER", 0), -item_bal["FERTILIZER"])
                elif op == "PLACE" and arg is not None:
                    item_bal[arg] = item_bal.get(arg, 0) - qty
                    item_need[arg] = max(item_need.get(arg, 0), -item_bal[arg])
            for o in a.get("market") or []:
                if not o:
                    continue
                op = o[0]
                item = o[1] if len(o) > 1 else None
                qty = max(1, _int(o[2]) if len(o) > 2 else 1)
                if op == "HIRE":
                    day = (t - start) // self.cfg["turns_per_day"]
                    hires_by_day[day] = hires_by_day.get(day, 0) + 1
                elif op == "BUY_LAND":
                    extra = quadrants - 1
                    if 0 <= extra < len(LAND_PRICES):
                        budget += LAND_PRICES[extra]
                        quadrants += 1
                elif op == "BUY_SEED" and item in SEED_PRICE:
                    budget += SEED_PRICE[item] * qty
                    seed_bal[item] = seed_bal.get(item, 0) + qty
                elif op == "BUY_PRODUCT" and item in ("WHEAT", "FERTILIZER"):
                    budget += view.prices.get(item, 0) * qty
                    item_bal[item] = item_bal.get(item, 0) + qty
                elif op == "BUY_ANIMAL" and item in ANIMAL_COST:
                    budget += ANIMAL_COST[item] * qty
                    item_bal[item] = item_bal.get(item, 0) + qty
        for day, n in hires_by_day.items():
            first = view.hires_today if day == 0 else 0
            for k in range(n):
                budget += _fib(first + k)
        return budget, item_need

    def _budget_guard(self, action, view, route, step):
        """At every block boundary (step % 72 == 0) make sure cash + the value of
        stock the block already plans to sell covers the block's purchases (hires,
        land, seeds, animals, products). A shortfall is covered by extra SELLs of
        unprotected shed stock, highest price first; SELLs are moved in front of
        the buys so the money is there when they execute."""
        cfg = self.cfg
        block = cfg["block_turns"]
        if block <= 0 or step % block != 0:
            return
        budget, item_need = self._block_requirements(view, route, step, step + block)
        market = action.setdefault("market", [])
        existing = {}
        for o in market:
            if o and o[0] == "SELL" and len(o) >= 3:
                existing[o[1]] = existing.get(o[1], 0) + max(0, _int(o[2]))
        cash = view.money
        for item in PRODUCTS:
            planned = max(existing.get(item, 0), self.future_sells(route, item, step)
                          - self.future_sells(route, item, step + block))
            cash += min(view.shed.get(item, 0), planned) * view.prices.get(item, 0)
        shortfall = budget - cash
        if shortfall <= 0:
            return
        candidates = []
        for item in PRODUCTS:
            price = view.prices.get(item, 0)
            if price < cfg["min_sell_price"]:
                continue
            protected = max(0, item_need.get(item, 0) - view.in_hands(item))
            avail = view.shed.get(item, 0) - protected - existing.get(item, 0)
            if avail > 0:
                candidates.append((-price, item, avail, price))
        candidates.sort()
        added = False
        for _, item, avail, price in candidates:
            if shortfall <= 0:
                break
            qty = min(avail, -(-int(shortfall) // price))
            if self._add_sell(action, item, qty, cfg["max_orders"]):
                shortfall -= qty * price
                added = True
        if added:
            sells = [o for o in market if o and o[0] == "SELL"]
            others = [o for o in market if not (o and o[0] == "SELL")]
            action["market"] = sells + others

    # ---- layer: room_guard ----------------------------------------------------
    def _room_guard(self, action, view, route, step):
        """tetsutani room_guard: at hour 23 the end-of-day drop pushes every unit's
        inventory into the shed and overflow is destroyed. Estimate the shed after
        this step (stock + carried + harvest/collect - feed/fertilize/place + buys -
        sells) and, if it exceeds capacity-1, add SELLs preferring products with no
        future planned sale, then highest price."""
        cfg = self.cfg
        if step % cfg["turns_per_day"] != cfg["turns_per_day"] - 1:
            return
        cap = cfg["shed_capacity"]
        units = [action.get("farmer") or ["PASS"]] + list(action.get("hands") or [])
        carried = sum(max(0, _int(n)) for inv in view.invs for n in inv.values())
        produced = consumed = 0
        for i in range(min(len(units), len(view.positions))):
            tile = _tile_at(view.tiles, view.positions[i])
            a = units[i]
            if not a:
                continue
            op = a[0]
            if op == "HARVEST" and isinstance(tile, dict):
                produced += max(0, _int(_get(tile, "yield_units", 0)))
            elif op == "COLLECT_FERTILIZER" and isinstance(tile, dict) and _get(tile, "fertilizer_available"):
                produced += 1
            elif op in ("FEED", "FERTILIZE"):
                consumed += 1
            elif op == "PLACE" and len(a) > 1 and a[1] in ANIMAL_STRUCTURE:
                consumed += 1
        market = action.setdefault("market", [])
        planned_sells, planned_buys = {}, 0
        for o in market:
            if not o:
                continue
            if o[0] == "SELL" and len(o) >= 3:
                planned_sells[o[1]] = planned_sells.get(o[1], 0) + max(0, _int(o[2]))
            elif o[0] in ("BUY_PRODUCT", "BUY_ANIMAL") and len(o) >= 3:
                planned_buys += max(0, _int(o[2]))
        shed_total = sum(view.shed.values())
        fillable = sum(min(view.shed.get(it, 0), n) for it, n in planned_sells.items())
        needed = shed_total + carried + produced - consumed + planned_buys - fillable - (cap - 1)
        if needed <= 0:
            return
        priority = sorted(PRODUCTS, key=lambda it: (self.future_sells(route, it, step + 1) > 0,
                                                   -view.prices.get(it, 0), it))
        for item in priority:
            avail = max(0, view.shed.get(item, 0) - planned_sells.get(item, 0))
            qty = min(needed, avail)
            if qty <= 0 or view.prices.get(item, 0) < 1:
                continue
            if not self._add_sell(action, item, qty, cfg["max_orders"]):
                continue
            planned_sells[item] = planned_sells.get(item, 0) + qty
            needed -= qty
            if needed <= 0:
                break

    # ---- layer: clamp_sells ---------------------------------------------------
    @staticmethod
    def _clamp_sells(action, projected):
        """Clamp against a sequential stock upper bound, retaining market slots.

        Earlier BUY_PRODUCT orders can fund a wheat wash's sell leg. Their full
        quantity is an upper bound; the engine enforces actual cash/capacity.
        Removing empty orders would change the later lockstep market races.
        """
        avail = dict(projected)
        kept = []
        for o in action.get("market") or []:
            if o and o[0] == "SELL" and len(o) >= 3:
                have = avail.get(o[1], 0)
                n = min(_int(o[2]), have)
                n = max(0, n)
                avail[o[1]] = have - n
                kept.append(["SELL", o[1], n])
            else:
                kept.append(o)
                if o and o[0] in ("BUY_PRODUCT", "BUY_ANIMAL") and len(o) >= 3:
                    avail[o[1]] = avail.get(o[1], 0) + max(0, _int(o[2]))
        action["market"] = kept

    # ---- layer: dead_stock ----------------------------------------------------
    def _dead_stock(self, action, view, projected, route, step):
        """tetsutani dead_stock: stock beyond everything the rest of the route still
        plans to SELL is dead; sell it now when price > 1 (on day 29 everything not
        already in this step's orders is dead). Highest value lots first."""
        planned = {}
        for o in action.get("market") or []:
            if o and o[0] == "SELL" and len(o) >= 3:
                planned[o[1]] = planned.get(o[1], 0) + _int(o[2])
        day = step // self.cfg["turns_per_day"]
        extra = []
        for item in PRODUCTS:
            have = projected.get(item, 0) - planned.get(item, 0)
            if have <= 0:
                continue
            surplus = have if day >= 29 else have - self.future_sells(route, item, step + 1)
            if surplus > 0 and view.prices.get(item, 0) > 1:
                extra.append(["SELL", item, surplus])
        extra.sort(key=lambda o: -view.prices.get(o[1], 0) * o[2])
        action["market"] = (action.get("market") or []) + extra

    # ---- layer: terminal_liquidation -----------------------------------------
    def _terminal_liquidation(self, action, projected, step):
        """fieldbook _terminal_sale: on the final acting step (>= 718) replace the
        market orders with a SELL of the whole projected shed."""
        if step < LAST_ACT_STEP:
            return
        action["market"] = [["SELL", item, qty] for item, qty in projected.items()
                            if qty > 0 and item in PRODUCTS][: self.cfg["max_orders"]]


# --------------------------------------------------------------------------- factory
def make_agent(routes, router=None, opponent_plan=None, **settings):
    """Build a Kaggle ``agent(observation, configuration)`` closure that never raises:
    a failure inside a layer falls back to the raw tape action, and a failure even
    before that falls back to PASS (with hands padded when possible)."""
    chassis = Chassis(routes, router, settings, opponent_plan)

    def agent(observation, configuration=None):
        try:
            return chassis.act(observation, configuration)
        except Exception:
            chassis.diagnostics["entry_fallbacks"] += 1
            try:
                step = _step_of(observation)
                player = _int(_get(observation, "player", 0))
                tape = chassis.routes.get(chassis.players.get(player, {}).get("route"),
                                          next(iter(chassis.routes.values())))
                if 0 <= step < len(tape):
                    return copy.deepcopy(tape[step])
            except Exception:
                pass
            try:
                farms = _get(observation, "farms", []) or []
                hands = _get(farms[_int(_get(observation, "player", 0))], "hands", []) or []
                return {"farmer": ["PASS"], "hands": [["PASS"] for _ in hands], "market": []}
            except Exception:
                return copy.deepcopy(PASS_ACTION)

    agent.chassis = chassis
    return agent


import base64, json, zlib
_DISTILLED_ROUTE = json.loads(zlib.decompress(base64.b85decode('c-qZfO^+N`a{Mnm&!Ow1tGj*UQoWL3N+Xbz2Wx>?EZ{W^80*8@H^cwkT+VcNRb^yE<f|S@Ck}dsGga?1Gcq#bm;Zh9?|=FG-~RgdH~;j@o1fl({P^bc!<+y3%YXgbe}DPl%a?!q%isU=umAJq>tEjd;pad7`KJ$$-~9aU)0>AkPv4#2e);$K`QiCHAJ6A+zI^<{`TgUMZyx%~$8Uf9{)e~kzkK7};}iP$<9FxtPj4Q&_HTqo+W+G_AJ6aK-#$9TAO7R{!>8}xfB&!N4_|)QVfJ6<-{?Po{=bLC2z~$VpMU=8@;Y5QR^WFX(+3{@@tyC!|8RceelZ*H`5P`@dRjmHAAbJ+{WpL9GRdEQ{`z~a&pvyv`RMWwFC&os+}E{ate}pM3;fQ=zq1^N(hm>G|6hL8+w0~1+jnQQTz+};<Ku@<-+i4;^wrb(<EQyY?m&X)bVq%$Blx!u{P!*&qAUL}7}+k}iMNTH4eoXL*5mjPw?So$kKl=S>H9Sg|8Y!);k4t6(HQd8IKF$kJ`3XbTvqXXf8WOQ<<=m>xIC)5`)uAW*}SDiosG)NvYDTdU<_Wy4lTFmFW{x%Z42WymcyDYnS2Bf)^LYE?d;A!{^QXW<98h^wh`gd`|;B?uAl=tqz~6R(0#+s?k8)9_WSH<!&si*uGiwn$Dh~N#QLJa3-;;lr!RlQIq1gnW#X>x9-UVc3SVGkS=hvvl?E@})xs)VzR6=cYsA@0ef7P?D$V|cKe=*m-%h>f^|e}<t8u=3^8EYtmabl{ofX2tHM@G#%RXCP1L?9^z1VU*(9X*aE8|Id))j|q*ERz29=yF0eci%6coMPDv2yNs!4TYU^w-b-$6oC0F@NHt9^lzAbm!f4C?`T&N8O9<2Qi*QuaBTOFk#H>Z7J4DJ;Y$AyuKW~WjrN+Xi|Sj<^$f2U*DeYkc#vl0Vz8(ZC_`MD|Bm?xqzLP9_vmfK2-uD@a!b=oK%Po;IC<{{P95#+${iQw_mzk+_kYE8?Ygik2`VKUhEWfg{HXSPLCnT8P~_sJvNhleBB}PvQe>oFu<DGv?J9$Q@Q}t<=_l*8iJL^a_E|lH3%`9A?dM<1;#-)Z+!vnWDl329^Pi%7e@5<MLg}6KVxrM0w02-yx(y+2OpMw23wXw475r5_B|g5brxy(gL$d?C`i?cTE}Ta((WRcQwh|Z#-{a|d?9yxGlq(AoqlJW9TJG(@#^FI_vd$?{yh7kN>%RpsYR(Rd6dyVd&bh<1c&?!n75h=ubkw84sM}s@gGl(DP2GLyLAQ0782<CNbLRfDlm5EZ6-fCte?<c?fFpKPqT0g{$M#^O+(UC>lpL|e5qWg?Xc0Qc!WiOKN;IK?r&U+hsygkzqu9vu3!6`505`(Y30lJt{-jPxMO*z9b4a$&;3!=vAbkMuJyC$FBr~OF9>u#u2qc{oPZm`dFZ~2!!-(!c~x2~pSR%5829DM)Goiy#KoaYT3`XqZ$5OC=%}a-YwXM6+<0u;x1(RKVqn81Hhw;V+xFD4%sA3zD>D>&XiY5j7`<%C`AX7*RKhq@0A*^g%BTBmSGxKmd<W`a*Er~MFhAc}a&L_%i;yACSp`6r3Oc)nqM00^>$qH(()y(88yC}LoKD4%XEKhV(zkPnVJfUG9`G%H?+KhkdE7r8TloadA!BJ_W2~v+Xg3Pb`LZ}!l_r*}^sV%eBVViaOB-=h7bR0nlevuJ^Db_rp&lw5Pj1$IJ-!_ZP|oG=hdS5=o-}~BmIzC5RV3HlTRnxHnonpN0c)f(5*Un;TL9u@qq#BLp_wRZ#k1(vKh_KWMq$E4%+2so)LQz69l38F>8V6OIoiB&8$5T;Ix-}u%7X0gQ5)<^M1l^+3$Hp_zh=mBymzc}ED<=SdyVAs_Z~}AJmh<5EV1gw))o#z?60h~Yc5f&t=8#UQw!q+1*?^+@-0I$&+bwe3OfV(`jJr`oY$}C&954_d+xPhoZ+sOyJ`(@HIv6mVsmN^diqzm{SQt&L^5Z<8q!E8d6a4iWh!malgFP$XhK)wd`DzU)%)_Uh^#t;_r<OucpQ8tKe^;qr$nhua>Tt>?IpbA;{~@3oEKg`X>b3oRPg9L!Y!+%?kbUQsZ#hd@@CVHcWfO{3poMIP+@r@S@TAi$Zys_!?flEHa0gMvo0?CJUmo3;?UZd=<VM)bk_=WT8gASE(&XEx@bGI`?pt>m8B{}cehe^76g&%0Bm=uG)vj*Z2))2@lK8Gy?Ij^h+@=N@VbkOfok7dhx$-MCU#}e<Nk*p=hKs=r623iQrI1=O?#lFOZ)2F8%qp?bl<)G@Gpg?*L>TIeajGSj&(<zFDwuLkDorgefq=s!-qdrn}u)dr&5vYkIj98X(f9+l>#sKPsq>iGK?W{D7B<_V@o{S!OBjl6pnh~yAq)n?svUW!LED&%~vhh0I8vk*$@<wAJwWLqrph_1*<EUjJUOC+;lqHyRi59iw?VHs%V#G4#jJ6=$v{v3O9mo#O<uR!jji@*jN&2mLqB9KA}B$GT8d+cZ61q+YS}ok=Oj|vN~s{s57k1Y=NOScx4g-EYaU;adLmmhzau{O5{&s`O^*XO6kN@f|$m8Q3Cty?y9F1*Y$V(;<~%-WTGq99j4Z7er%&7cZGhOgGTUf1Jtz2x>DxUE-7K|wh9>mw+qU8*7cgb9d?)Ts@xz$QDkjjs;c*!w{m|nbYkck)u1TQTcVbxM!@iD@_kcI*lr<HS%Dh?G{(E~&g%3HPyYl4D|ai49$KoQBluYf%3Fm@4`Aqh?)vs3U`YXTbcsgyD6GNT`Oc-M>tDYj-hNuyQm1*HM%Bh*NE~@^()hK4>ld=jmt2T$Yq`#g0l*Gq1JeSotPm7L_W-1QeL9pYv7Wy|$O|?YsDedab48&m67}bpjc(N&+umc{9@#i(CgvG=N{9?@RNp9a2;MwmJCWbNUP-C?EdAp!k7wA(_3z(ipumtXCcH=j2X+13o+abS!LjhqREQtNM;8O7X^xH(<Lyt(x13KI6cu5(B7#iO>a`t4Uo!U>v=Y7{%0QQI%>F_}E~N0_M)Qp`8+whUBzMl8y;}r5wM5W;%YWL$9t5XEt}MSnnc9g2Nl=cSNrp>)d_KHV99|i~{`0RDqdN&gb5y=Aq#L2?T^LCfaS<KS#Na&UhqN5252)4YMxGzkk;`;7H#9d`ILOQ+-q&E5z(X77c(@-BO6HABo96ej-+vW4+}4Ky?v_gkeeFp@tMmihJGS>iHS~^q<T^a@k6Q!QQp&w;EHoZqV*Rw5=O$Vd)vBeg5+?l2UPZ7ui=p@1D%`UUK$We`_f-7-@vLkK39oA~{<>M7ffkMYYe@7x4?Cd}t{d}bBk0HHjl)O&fHOZ$!uie^u1~nqA9eDa>+r?bG!{Ow99ljpiTgL*U#kZ=+#gN%qveX=cly_&*m#gtkue|en!(Z6YVvDg<V=YR|Ms|I<;%{Fs#NdF?Fej_)N;HU=-+Zhsp^b_$@x<v=2mTA#SH*271n#w&;}}(ywfV%Qpi|`lg_iS>%|kB&Q1q*i)^mrmFsmaBOJGUoNT>hz+Mh0;HKH8Jwf-H?!r#KJ&z6pM$%W<5(uSyWPumor@y~z2@&LpJ-MCH{ud>*{FeBy4$mnAoX!%KY6LJ3hzhOGokdm81S^yJa`sURwi%(lInPvF_n;Wv@}!r+e0+S*Xf=aYS`|BtVS5rejQmG5Uu#U#dD8@rW0f`mIY5}=OUn4oo6oJm%I(=^BNga+0FqrF#bmiAE}tI0;-rwbq+i7MNp%jamTgnd=J>iSzI}ZBc*aFvQQC=k7{Fv_#j(p|U;pAmwa$<X9Jz(zTGj6!AC=;0>moazIa*sl??5zs8&Sbn=+OuA4T`{{Pr9b!=B>06#A#|hyNuW_>b@10>+N-)_r$x$jQ?`KMVweGn#b8-j7d6)4`A8MM+vQi+gFHnw86<KJD?Xcffl^k{PJMoAYhY6IL>NWeYI5|XUg;qT^F7fBZhW~T<e6NC1|Yn;ZTUuP=Ob|7h5`Dko?N2e#YzHSsgHH=~6#fUc}4xsnXO`#RFs!8jcVmyO0D(g{9^QZZMh;P5)RVUjZ`WRm}*a#P4ghVnBo<B4%}}Skpc0c1>fuunGXP3gib|S*MvgxW2I<0FaF`S{ahc8>c*@(+P(+x}k{lw9owmSlAkOG-!yR%C!Z~C0{VMu*TN5qO)F9w-NYC+Cj=`9bo3e^A<Usw6sjL@YyAvqKIn#QBf{Int$4hE6b{3CHXi*Cr;KClaH_Q(5h-w8vA{H^;RNuD)Q81Ofxj72o?zL>pi!prN7BcOgenWfrbTH_)$86kmd*)nO$40epBO*2!;7L5!skh#VrkEl349JMy8)j*Q{}>Yxe2h#(`Qs-b0&!tbgsHwC{|S2C&9RC3I)|g2a+J^k3o`0n!RcoFFoz&@Ig&X{P5qAa0E>j;qo68@oo;$LQ=!ZVvd<S;iCSzATH})V`%I%@B~wrGUQj<#sFOX?l=k$Yee4Bzh5aLF8RfC4x*!nZ``S^^&PILZyNs-bZaOwr;S?;PKTr8{aHuck?Rp7ANoCAQyOkpsz}Kh_!dlRu?{vPJcY#eLsBv{-4)e58u?XYzLuk$+j#`^sW^l!a^2WvYVOim^a1i8PhQu6Y=$sZ3+2%N23qm7K%Q}wF+WU1T6?kPY`RB#gi(mfeip@sn-vUwy>Ng9kPP(aAjm^tSyk~TQrK<N#)AvjMhiPxG7_amX+5p@<vcaJa{^9B~M6T<Cz<zMTEdB$A$=iV;B{BVBJl!BBD2<%8jPEI86}+qL$ik>UyjZtG5<b0kR>eZtz)C0!6HMpy)By1EGl71}T0OIK*^5!o23>a4JQkCXBU=(8`K=$%cFnDDEdoGVNyo+d8h3)sjNgwBlOn$?mr;*zOUe#=vCtvZPA5czseY^qM4^wo*7UP#-F-3!{jP(~p!6PrWBCp5;-%t8`W+Ov*E^^;em$1%Mnxv|vGnPe7Crf5>(}Xk%*t9a$z<1J_KZ!;n7Tmj(Ud(R6`Os4F{y>3WIJfBEs3$Me^Ieb|u(z};3%YMfDCFdo~{LOZjcqiZI~lr}7L)@aUxs9a*uycg?LOtu-a{&BT`T@B>MTH)^i&*Lpjv8)8~yeH8lBy=o%ylN^-_=1n%IEd~iNJ}Xkv4yrIeJ$vfMmnUfTOmdh@`cqrvr@&Xo21ad$mt27y`>67fp|niR@Ol*UiHwNzoel1fN`mU{20e@iq`4X&aG_bVln;t$qly?RYov*{xVJa-PRxLPH3w)4Nw|@yiQ>%{j)^cL^$-_n7e=oZr-?tXB{OqsCTX9#C&L0c>sb<Hew90W3|KSBw8&<tG2S+Uw)f6*hPdit+*CxwI~*d$n%N;4b^ZVIxSDHdJGRRD#o@%!nsuTuTiVjmTq9UJ|l_Xikx`bc}roLQl#sCgx1#9^ogJYyJ%mksm1duBeK(KAvhQ|FD1%iq=_r&Q3VmEPhkNkMsqa^rP2kxGGgBff1d$JL~t}X_E>|K8~#+_W5+2?<~THwW^mbxnpESy(eo5I$HsON#@lIB<x9(;Nn5CcyV*z4@pIFtfr&&l(KyM4C?Fn+Z2#Ok_$Wb+Vyxx8EjQQc)sYy+C5G9!|Dh?;0&y5Ys_H5-e@t;zF-}aKZ;o5A(IKnF1j$%%!bE-`0OROw+t?^I5|Fxq9!^lhhhDC~XHLw}N=lE7Me`wKy*c>!7b-y&`LzY<XYlV}0^RvjhK*UUcSO9Sxt#DzfKSXJ#~L{<-k)jLQ0_S+!xVur)S2Bedr5tEoCgH)D}(WhP=`|n$_<{_Mv6Za7nGvbu{pTplXF~3F2QGa#Emov2(8l0hp2`#<{@J@%Qr<t<x-~2qSo}{9F_}JT-mFI(8P`$junCgEvMRc6Y)N_9lPYjIRHuAl9UV#xhmNV_E3cBmV(UR-6%Gfxl0Cg!gIUaIsrF3qHMx~2>Yg!pFT1vm`9X|BE>Z@n78a*e{Y^w6=XNOA2fMWonJ#U?cHEk=KTH9f5Oq!e6d}B1KlilXNAs=S~Z8L+6n4T8FnS!%je>88caq@Q+vCidO5nY5l<*zLe%ezrOi=V$i%H6a{;8>;Gy<1#?3LFfVPAMO~PDGZ3LvwaL#u`Q_U^)ib=kr>o~P7xhK5U4X=)G5g(>9sx{Q1o?h@}Rrdb!$@$xF!3UjgKb%TN-CpC*G&Tj1vEfIy)7jNFcyI_=1|0AA!p%J+Dhz#715nh&;?-UkKdr<vNB*64pVI>!x>b+ZhGLHr8i@Jt3<8*+!twjXkXG80Djtbi=!MI21=FxAZm{jsZOcS(Re|axDCH37kL}7uq4AKs&Ui51NAO<aVuH?EUJpRQK=Z(ln|}AXA_fuwOrLtlJGV>^>MxJ`koSDcNs7n~;RB)QB`dtr>|r$H@CcFnl(<Mv@LS34Q!AoC0ub*@TZF;m8t-q~9oCJ4dW+pRSY8yQ5<t2<D)+B2;h`YHY*Q?5x0QyLOaQ_B6tTp(<?&fiXyl|=32vKAyHn)VdzUVXyNN{jO*;B8yBER}Z};V*Gjd<d?_UOQ@KwI1Hc|o}9~hJFWKVN_0vdVu|Cvl8Yh9Hk_wguT?p4QL&ni?J|2|v60SAmg2a|lwW5L**0`gX$FhEOO=&swf_qc)GpO6B`+7aqcDISd1L|@ScK_9ir-mV)tvtuO=MDjl>7Rlq3>>XzgvkRDR_ez`S>;;|ua|45Xpr=CB;f=-AzR2uk%wVp}`la5}*mb{6Q={x|v-Hb)>B3V%7U#IY^Cx{k5?d^_Hd=Azppz3jgB2WQn|+oa&SG*dxC!Lmj+uu70(p7oos$n;@ThAmnZmvalMuo-Hf$>$?8Ykk;Y+frL>$j5#S$#}a;><57W1xMQYJMsF1Dj0CYbt35QkLK;7k$O$}b&<uoeYhMmD3%Bb>#x>RmF}3%X&bpm2heU|tqca1aRj+l}d?FRn;sT2-)j=}STxg2h@PD<q`|^NP5+r&cBaktGY>sbCeaSN)yK|72h}iHTHL9T$dOGZqp#Qmp3;uWv)0FOiV7P>qAp+s}aWf)s(VYrzlKNSKY4SJv>MBD^HECR?34p*quhuqwwATZ=nNbQ>^l^^Ep5xM}nJf#G>;te?2gK!U7WTxxfkuG$uudz1-hu#`@DL9N@N)T!vM9sL_b+C$Hb&8*6)ISi)Vx+-B&0Nm#HI_@gOGGeh;awcO<NhKkAl|CcjQCN{s-&lEJ+)4zi^?%ex2;fkOLJOQ(ozvwS_5C4=2{&p~5(EAH(y)Qlfi;lUS3KNcSpF(BzF}L(>R#e%C7kE$3Y{QfFK6&24XvXuGA2lQ5TUfeF>vxEmqb~Zt+dPUP|YL}d?Qzj^6^^LrCV%%7;vR9hPqa1+?JXRM@T&e2m<d!ml<;lxlLxA<$fioV$wmnwb{2xl{oSzX$GFvx>{4da!tQ0Y5StqlsrdAaqaSj8|fpXVI+`X-UYgi(&^uold=YRunhJuGFQf3S!B}_B%!DoEC*D!LEbP?Lw^$+8j_!(RlHF(;JGMM8t(gB5!Jf`e2eD7MdwOO<`$SS{Xx5@`QE10r(MDB5}>Uhb7R+U%v5TNKig)ys}L{!1T3>n2M^}CS)lD}s3^n0_^&9nwD?WZPEp>%T_Hfp>16!7%)fJ;eHE0|@Y11#HoX3y&W0z`E%)<DeZ^#sUr`8ronxx410QdY<*VYL=9rwe2AUIDe3odx;;65fxt0)`Mq`WSl~lbhr<0{;D4DT-J!7Yqp@$Nlv>BCvYFkY}K^-4#UCwloNSf=GIRl7(WuHvOKXa&NTbd9TOKb?Fvw)XjZs~*JQDD=h8i6z6v$&8In`%TeyH~~hzqG=7$JQAP{efL)D&SDOhWcdQUF{tdldfb4unT+((L5L!mHv6VmN?Okk+n|XPq=7%r`6n<3d?f!6qC=PqS-@KvFK5@30Az2XbV79vFVx0m%BIX=1%dxyxA!2AQvBE$*|8>7`r>n2<Wb*gKS_hMsc&N=4^eb_XI>q2oJD&Yuc^coP>sonw(q82BueAWK}QTo3C+J?BMM@sdlSnh*H5mnM}3S@Rrf0TsbPYi>iPaDHs>Mv)8Uk&i#_^AsbMiu>z7@@ajcJ65kS%S5Pp+2>h2^G9Qk1AQ9k$5M<*0QHu{ypL2CLU|qX}XPKY+K$9wpThLZ5RVn56TTOex!K%<R=iy>@urO`F8G405VNHCB+tzFC1Iw7-p)Dlf5iINRSksDQU9heTtt-$u->i18C0Hh`B@zrgGnR?+xWwWjLKK!Mu$hy2?UvLpf%RCfM3HG~UBWtO_Ol(+LYD1LR%HQOV`|gRKU{FllcacQhyq9HvglDIMGOlfy#racv}TE$xFLXOQ>kK}4R)L?_HtiOi{dv_?d4>1^w76~F%=D)rWRKwYwEBta2J>a8eM^|fueK|7}U$Dj=aYs-gI^g+7oeV0P<2T$$TY)agtZoZRj%>T}9`xUJW_{q%Dh5?%5()uY>GT1P7M6h?Qw90Vk!#D#_2MJK@`k9r0{!`0nxA8;_YyStUHizm&1#`nckLkW_s%EBF+INUqbn8jwqUU>*#V7ww92EClG9ltaN|vfxl2I9{^5VU6!#<2z*rOxcAnjn!=dNxHF}BIOQcFjH=JY{ez5)bOV>mM>DAGCwvW#~SLY)kRcrWxqA8Y^}ORB=0IVYw;dD7MwD~eUq=J{5)Qt0OV-NST{n~X#*Qc$|OOmt?;VMc}c|f$ENvpC4B%|0%=;F5sS;%M!u5s#sYvT>ZZ|iQ)BXSrw>lsdQJ^Z0yH@T@euWkzg22I_lpsVZrmiuVAuO1DmI42*A^|eY3osb!zuO1&Qn|pHnz~-R*MqijKV=(Vb+wyA}-q6^#+u{3}v-PHGZr3z4(RQN^Ky+r6&LjELzX<$6PI_UIC~3Qd{lrx3GG>&3Usl-B3m1TRUjyL|Y6UJEYvc^26O7nokRr<^~lOaADW=!f!+?JFtj(3fGlrz*<m|%FvLtUx6s!sve{g-vo0K;&l00Bdi62rG6@Vzt|9V=Sz!%V+XJ)V_|{%3OWJ}Qq_F<O8fntdf7;ggyra2)q5SAmaK}|nNP&ozy!tW>60mH6)aV~(3vpJ)W8eDco)yDF=>vQ(lVOp%kKYh@FIq5xtnkvRRU|<eW_!&;2XY$rOAETL7YNtSJ^TiJ-npZ1z73D7ceB6ZE?RcgQ=P?`+dK0j2It>@j45AnJ6na7s5-X>DvQKjy<7BbDKw(d6_9MKfg+Ese5p0e?<y3`9aRtH~E1hJb|j1O=7tcbxS<|xwNX=AXArfCm*<Rh2O0L4@Wuj29&3g0lXE#DITv%d3QTc42kI6x;z&C)QHDe%YsVBu{OR6a_09XBaUi9r5MMo)~pp+3=1dgc>|`i7p3SH{IF8N9*+(ASwgO<NkIyjAH<dMw28$piTHp=-;bpR0D}2}jT+YnJdxvb%NFFST%T!@t?1b;<-e(c$JUEAiyVuJO|^r<OBdnG_&cx;4p+J_N+^RtQZMofA3<ROYn|+D&MDZd)A0@n>0I4+-A7GiykzcfcQ({Ssc)yauLTs7H(UGe$?9ivScp{1t^{<_gu2oIDUDS83(q5%Iy76~djloU3bjcC6H9sQHK?00TVOq}e(#qc0Ti0%K#B|ip9=hLdDP5d^r}}+uDWz*keSLx^WG{}=;cjoWLq|=hLR;nd+aO=j#6*xO{=8M-;Zr?s-DO`N?+5a3e{U|Yd6A1uMtS7tf*_NT@)D?bl>2kMB<9H5K=%lqed$}#aJ^wVwNx}J(25Kf+dzdb<gwI)d@T4OR8GXurxbK4=SJQI|?_zC}k*76O{vRT1h8^wB8=I?iK`*opr)(x4Klg$%<o+CSaesj`bE;$Q?61O$rT^xd7+hpw$c|(4|Uh=2{Htct}d)7Q!;&@p)9Z$WP`VT1;UzgqoBnxLPLqjFdTzTt#VZj-?i!zHexkbmo)^17>2fAcMW3c8r3axeCxldve*e@m09_E<k7*2@>U;x>>w0J!NA0u0mo^-LdcDdm@a}@1Kx4RtTA}S^cnG#<2^ZOn`n(>m#0t%%{g6-hO&~dFGAIpLH`^^*_AQT7xQk&<DJ#R8uCa*&)eb7t;@z<5%?Ruc}zr^9Gpy)RI{6qFMV1?Vqu=grlwt4aOLWQc9rFne{ZEUnXc%N2aAKAyltK^{mac*%dgjg$qz7ch-zdJdsWt6hi!7j#8`C2_D$DdLrr)%0-IV7&~cSkB-dZ5@kAmjl={LgVlN^E*ow#2y<mp){|OfN)(FPCd%Xbnx-%c9F=cH1$J5;G}#G(U52lzQ*$+`MVrtXo7lNJ+y0x1ZiB@<X)z9uby8r$&NRN5u78O=xct1ty#mP%cW$Ti3AK_?>}B}M9yV!c#0(9B3cME@&^({otMzo25~mE@2^_bGbIH=Ch;s`Stt*nJmmgo6EsRv@oA1BXs)x$g{4#9Me>Ve4Yyz~TT?4&|cprwj6yY;u@y0Gwwb<ja^Ssu*W5+F;9I_cNn(_!UyftJQC)(h9+|KSQH#a+ptw`N7i+As5JZfnlpR{jwX-dZ3JoSeAj9}w}gU@whdLr!R`P)68G*%`SNuc&Rj#NvP*TY(Alw1HK!~{g2_hYY>ha(m2R-{&8dlIB45(3~bBia#{p6spqpkY7%#;nb~Vr?4op;b}Py?y#U!>kBG?em(Xr5`$feSmx1zx6$~IMjWY!M$2#OnT<hw-4YEd#bQKX>5;-acnZE=wY>5NE$tmY{ktxa!I*dd5*R^l4VmxEHD=P3QdMcsyW^z8Oij86n!fL7}w|Mx98V|RC(8x$7svL<&@Hv1-M*_0!8<8!ucwg&Dy2FgS+JnQO23Z0>zvuLkg}~#g(+NP~u}rpQ|(nBgp_nQC>0^#VqSAJs4rCt>8%6*2S@a;KtVgyax4?7KWVqlNHLC<%yva*35QIt2VplkcQN36eC^YyST-rfl-Q|B<Fsp>UH8Vn$b`_7Z;_5%WJx`!lBVM>WLjYa-0?KC>vs+O2Rw`zvR}8>-l7=R2xTHUaqBleMVi*4h?jcHP-*AlF>H5WzyJB*O$JQ%{6CDC2=u}Ko-9}kq)@BYq#p0HQa|WjRD-jPCR0PQ_-w*v<qWZOR7JKx;ezK3Z0`PGvodGL9PxrbR1G(&N=CJtKY<ct$8^WnR#GD-M}weFSC3g0lYHp+X<|+*3Hb3m}L-?&SWe8Wu+9FHF$R#@;n#q2&t|2YtcZDzX4*IvOtI0hn_4aI~$N(UH~Yc55hzAkbfviov!ZX@eT;2&ol7di}DfNq90x3VfswPf|h7&k9=>ZT?^Fwkd^2TOlrcH+fpD#+KwToc6}xr(de<p908{s2HV`m>j&a{UhOI0k8TqKRU8N@1RrXb^4a1N1WCeAR9wPpBBV%MyUc%QtR|nLiEvuWc*-a|WV(PAim76jQsxEWM^VT<gPnX0;kds2D<~$@IW2ZWQ#3%tDSdDmoZYCJiq_N6^sq#Ue4eZPqoPznZ1IFMj!yajgYlLJz9cHKQVO08n?SUvx>_<}QuQd*Q-C+_lhroD>v(C~(IuZk&FQChEeG#-(7AbhcL*ElBIfcHYLZ-sO1U#l$nv`m;XS^`RPhR#8jWsr%Q39SL~JZnQbLe4Z0t{FwHUW?TyRzKx7VOTTb<dWfyY=&HH}+7gtl;On=*<e(1JtCtu-Jm@dfLs%9HbQ`!M-%L%ha3twR@h!&ZP+@C3hOloGB9+1#UV!ETU#n?D%h^||)EU4>n&nFrd@rhj$h3~O)Lozqge(d)+u?aWV=zM`*C8E9$lrh?JU9{hr7*T(Y#5n$w)#5lEd*ZbM2=NVL0XDhh?vEQ>Rc`OFH!OJj0OH6KNakg}9akAdfl00;!G00>?g`B9fJQ<kZu4uWSsmIQwifBr+S_Pm9tj&JId9Ik^GIwhLg^bqJc7PBBaglH6!}`32EsM{hX96E-6wq`?Q%_qsKbyFF83#ozk|awk7$F;NhX7Hd1XS8`O_B2!;!;=faJNafb~U5MDFJJ(1ve*Z`?d_ORH(2-!BSI3U)hq$FM{NUpg^{Ccav5sI0Zq`UkFF2jIfr`Wxg_(>tMKp6ay>Qp&?;ZQ-ZCUz+;nr5#TC;3Ck!DwDS)BC!L}RWyBg4(_{X#Onj64@UW}c6xue&pkL4SycUjy37xi3H3a+b039_lq{T#F;vgWMHwL@#)=Ja$-FG1hbha_l^s?SB7qE>PH_LQ9S^(8~g?bVO>zTPp%8q^t17pG?t44$JdOR(%WQH1J*L1Qyq00Itl{M7h&m&b8%nyJK4T+EGWT<0Or4kV^G7_PM3}=x*e9r4RajP_j@miC{*O8k8zMA0<=EvGMN|}!M%(&}+PeerDAG}`z*hs`CM^a6a;@24pD#v?v2V9Qvik*1BR~9+w40%PzaN0Y|*=$Wsx)Srjq1O2MS0X#QloOn>wrOt+{T(@p%vsXw%+lD4%Q8DupzY2vf+}+!VG^4oXn7c|IK4J`CiZ+>#^msMyWJ8%4lkuy2gob}ac-(~taV&7<`l1!yNt_Zo@<S*mw~n0rYn|A$??lic8m8?q5Qm}(|U{D3lby@0i%{i*{70F6_<6j1DT_&l-<9rtp#CHafZgjHp6U7zyhRVD_tf@n{`|f?3Qh(;!+WDCKR=72TR)&gS#C!WCpqn{Kj3K%BR<n=DD^DGkVW_QooiT1HyuS7zsN%I?|p1XMFX&xjW*0OekbFAuZlnvg8^h(b$=A+;${TJPDGh%q9knQeS&7L6agoB$m7w<4osdhsbhcEyFQXfk32U6D}mXa)AA)>{<&ZQqDQbCLE%+z!xhDdNZA)HQHaM2qq2e*QbSV*C23NG>*+Xl(veLNZ%r|1JIg1TM|Ze-HX;UB!bu!t?(R&TNPxP!YagB<;nTUGn5>ShQ7LEp`XQuH2`ki7>{25RWU7H4HY}h;HR^~sQXRg8%F&s9$;vF`%-QIMJ6U$64*8=Dxd@u*~LIoG4GHUwXm|1(nwabwS%nkOBp$L+RlF+?a@#VWaQ%xeYp2=!8lI36O5HJ7U2xX$U7u2I#@}}ev5(3(mQ@gB)VJ;Z<=J9#O^ez=~@`e1_-wtf=T$!7j4?jF5F+)exA0W#cAD&{%$gnm)!=yP*8MT=X#nY7e7+4h){gIW<-eA%!g(U$)I7GJ4|!oszuh{R^{rOE$HyvTH|a(R8@hh&=Ds6ofypP_E2xW5y;#MB?;udBi_A5opMbWDlGXYDesYi_!vH2&D;h#ihZT@)o@0L?dy44(F5Jjyo8SoV^FzvXyx6a{Mp9>(SvywJ-L{QDv#CV{y<p9NTiC`wjwY!P@H>{PFVnDQ5K%BEWn#2memzb2fo@SZ8h3b*3mkbW>-^RvbwBMiAZ1I*od~PngWGn4VhFXPFQakS6aWUA3=5q8No{Pp7Jy*Se%9UHFaY8twB--1hEZ@S(*!8ZzL1RqtWD;Pqv<AMQ_%Dv?A{lY!;o1QfdH}U!{E*@v-2__YB`i`@Taj^*Z$zfAv*Cu@Z{YtasVf*rk)v%RdyLv}bC{7~r4C5<UrYbmaE4ki_Kp+b}ZW-B7jA0A>W0Ks+WP7MOD(9(=gj9jb-sO$VAaWu_uVY+l^}r$L>wYOMjTZDF#*<kFs(gga@%nv!WCG08gC0vs<FBR`y+HAD1`YAa&7=lUk1>l2mL+T?tciV<*Go2)^jjR|RAhrX^G%`;UmKZ1>bwVn<8i~cc(MbfA2k}BuU_MxuX(PXL!@dQE2QQQu7ivt4ZpL5F^nY%VQ3fQ1CSnYoEf$uqo3ECSwDNgOFUV)_(v!V=!U^LfPuu@^x1eK{NmISgQFwxZP_M2!MEaP6d-Sr0fNCrq8I<0j6R+(DPQA5#Ve9H#4Rn5&=-QH$jZ@=M+HIaLuH)w6Ybw%F^7<^7kcDXKolXJ0}Kra;mwBc+GYxb}qvXO8`ER9?rBtq4fJ^xJvL)uE8IEg8V{dF_T=Gw-S*$%}PNJhy_+J>5sa$6SO)`>_At7WaGH|3y@6-F^n$klow0-!BRMS|+m6`T?p?IlydDP^wKh<HBuk5-`Ul-Y&Xy)yj{T+N4CM?U=TfI*l8qG3wBX8nkie!Yq+efWdNL!Ce1!7~`@+YLcgjtq0jd;0WfSw<olKNjrGL5DtRTlX66xxVCP%j!bEs%Jw<B%T^^HfH%#bTchvM}|I2f4a95Jh!Z-c9Cu4URul)k1qf{HS=D}*;_K5q_a0OoAF6T<@SJM1*du<iV$VBNY0Wo14`az&cO3~6o?4350=%awxgHs3#en*)|9I5)Y#-gZ6uSfw6yalbF6JqK#hV_iH>9Ji`^uAS|D6f2T|@Nno=tsIDOgNR&CyRNrvdVuJf-rKGq196!2TYAO(jbv#lufDv!0EyDUy>Q^#bWCRC!c%W6tao}~Y}+x^MFyb9cAU5Os|jo>S#z9q1nR->u~G{T~QX;ECz@Ohf|Bt5{nbx}<m2kk+*ECkzyZHiX*ntY}RG*Uq^<f{FZeq0hsC#<Hu(Fq6^6Dl5IEl8vl8N9<Iudo5417|Wo?OL#B2JH$Mt5-1A*XZxJfUp?zueW&Kd3&wH?c1a-iE5>mWzPaPJ3+xn8$s=D8jR67wdNcd0!!=v;+pifcvBc0@l6f2Fx(Lgtw@35ep0sUn*^+$&RWQ#sO1iKo1nlbFw=KxvFY@OJE)<RY+_a3*ap0rNXx)w#@B5)PT^zRX474WXW?-m7lMW)Z3+ef{s6}h(3y23%q%WatPcK41whb-(xtMUS>3iA?jLN0Tv`Z^pikZIG_{tfiMkG6P7KhgI8}lHA4yRSi)df=c@Uq+xvb>R4y8eEDhpju6h2CJ%5s(v=B?We>SVtsj?wza71$dsB9;AQXaUsNy=B2PFB7c^$90azffVlbTA+0>P%xY$*7nq1s$lmdzcvuhjdn6I;NZhn>al0)FdoR=ssx{LsU~fwgU^G4r+OXeCOmuL=XEe%h{|VXr4ewEaxWz>gQq%I2kptKf~4$qOfk_^Q?7=tvOXxU-C3U<sHLT;CsnDi6zDKftrS7E)_t<{(W*4}vj*VnM_1Vf$R_}TwK>lE>G6lRpB_QH5^$<P^qX4tmGQ@SsD01GqGZ1j7)0nW5DUdkMA9VEZRk(TF{)!Gi}$PfqCb_O)=pbb+otH##-8axaC;Y@F?*iGLaA<u4i%L+2Yx%jufevv8!JZV><rMiWNe>P=!7P%Ams{XLv7cA2{62`IgVcsb!A%D2>H^%h=-<=^c{_OoNaPXSj2B$YvJ!{*cR^5>uibjmWEKJk}?VzWHOy%CPf$S1p8~W%dsD~;4kjNk(!cY2|t|tHd=;=OE0OwC}s14_i_VU_Yx5zp{l83z{Y&AeSoS~sfwXMrjQ6>@e~~h^0#cO=fB5ySC&lszLZwz^T>1$b_r-?b~>i#T?#-8U`Ddw&YtwFVS5&z-M43iOHExYLX$8B3L1EbJje1x=Bj6NGqdgBf`=d+Hy=XxC@OmKx>i`~u(T5E{l=zqrXVZKGPF^TI^p9VkT45ke@z>3WV^29FL8E#(_mneWp<%;leVeS&|JHqR65#uZ#!aX#&-d6R@`c(gHBso6VW6LM%{A091mS$i4P%88>69gp+|;>YrYyAytUT9hxWb>HP$#?4lp@mJkM2t%7T|n%h&l_JNFgE-|2`VO5pffNFm+Nt?Ii4M^sI`>_WxnwAHF4bL?J#)<ZQe<Q+FA4aD+sa74a3E!nIok%Gf-%YtPGZ^d-&E^x(G@%in%)oP6Bdib_-Ytg3Uf`UeCg=5>u!s?RkuHniUM|JqT0rJ9&`KjQh!!=SYKhRu0&F~I#tx1GYBDXK2$f;>#=~~9CGe<-ee!kMfI-a*jtp$}X4c7>S2@Ec(w_f#wc4|wRPZs!vMdNJTLzDLs?bzFp{-xU99(ht6i<)5~g9v^$ru<g|CPOo~zLo_jDL*rNHCFo3@+DjbR0Rco-_rrlnZdu?<TJ+mKT{7&O#ydZvBsj4r&`(ytn>5^GQ%8{n_tewG369gx-&&66AWIs*R4l4S+IV}x>;2+gmsruD$2~ac14(Kkn1>R(_2z7+iSujXkf5NVfjaOnjKww<gti768td()h#hz*@zj>rTGJ@`7!_dP<3yu`QbMWdyKKmLPaE}e*fz==+cH7OD~CT-1Ga#AD8WH=%d6(gOSAaPh5_-lcJ36t@{f`T(yy+ye6i>R5;D2Ht&mKLTgLcic+nP>M20rsi#&{qf4Qe$Wui^>Z}G_p`pEXYDOwECQn^X0@z-HZ8nyrl#5o<BDD{*mXk;Z`CtixLA_C8$~@Z1$--i)<^+-UcY5az&8S8lt4Jy%^@eaxfjh@~D7lcwH4|a1Zgbt=NFhUpMK@)1lMxKWZeNh>T$jQ%H}ODFlK9ynU{?3ZbpiIp^i5$0AT?u~(hd8FVE5GC$jOsvRQQN62n9c0XmRrR@Lh!1sNPIpmB!wpIVWg4q9!1*vbxes#!-xX18P5}SSe*&aBr?7pXNH`&Z99iKaU{EW}gk=E&WWjxlOr|5geUp?Hr5-(0nxGuo8bywgc6pPqt{2&GZ;`BaHJ_4E|a53)1VMgO(oZm5*jm;ErIPpZ_1>LB|L')))
_DISTILLED_SETTINGS = {'hand_align': True, 'weed_repair': True, 'sell_lead': True, 'front_run': True, 'budget_guard': True, 'room_guard': True, 'clamp_sells': True, 'dead_stock': True, 'terminal_liquidation': True}
_OPPONENT_PLAN = json.loads(zlib.decompress(base64.b85decode('c-rk<O>bOBlKd|`_d)EYzV=2-J+?5iDNxchW(~qHFtb=-F?;atZL$A-X_4Li`ekH9WWH)jvoj~tO{!nLs;sKa$jHc_|L5Z0e*NulfBo&^Uw*#$aQ*S)#og-S-+%qjzx>zz2lqez?bqM_<FEgH|MSlm-+lh`FCT7i-hO`b>0))U`(b;1|KI0#tNTBD_~pm#_U-XEmv?vnyPAIccc0(AfBVb*Gd_L3-R6(^xPAYA`Yrcg{=>U3|G@j<_mN!x^zO&&-@oO}&F*5g4j+H~VY~fsvD(as_~G4~KYsqO|96*@F;0H>_>aGgjDNWQ7W41;Jj{ps{`$=pkMyUT+fP4ynG5vOZu{}m^dmX@aNRG{seAl`&B>4s51l`K740G!nKz6#*Qd6HzJ5P-)3rDbhx|4U=+WeV+`hm0X<}%4oR9M`wJ-48$G1WA_wa)r(d@Z@`2E)>&yCNBGY04Ir%#9HoQz}k#O?L`O+Vgz4puogXFMOf>rdNT^45>D{cQf$`{%$-GJBQ6W-$)EIN!$=e0Z7ZQ{qtepFDhjeZBW{eQIdw`_3MBI!bTyH`>&X-$|cGOkF=?vW6q)*<a`QjPQ7SmQPRRwLbJFGyHI;Pd}QQcX>tUegc2Bc;YXeZ1Q|&TIsY89RS$gGXos(IOk#)6RjT-=Sz4NU^b1JV=J~d-q{buM;tF4urCfjJpL8A%D@yQ1LQ-VZ{M5u@3(J0{ql$H?WcF|-~H>C5hmVo=J+Pw;p8iQCy@-5UZnJk4u{xmNQuJ~&Ky2To`<CGM);Hy15#i5_V(sOW_|qInc>Zx>vi&gM_&scG^P1B9LL8uo&12_k8ag?ae+5YK74pr`HE}*6;0!Q7_{=ljL78Rc^`g2?__*+TpMsJJggEivgkv_&>*gaVPo<+kK-@xv-mBOeYPL>!xyuSHFMI?(S8`6JjnOA3;LGh6v}Ujp<(??oEP(NDE4ChxA@oRKhBp_?*4fcP(sE=ItWFl0R(^q4oWUb-=!!z4&bOf{7^WLBFvGzNcQpm%sSkrdSaQcac~8e_-E_8!l_H22x0^I)zQJQz=6iKmIrOVe{qUA$6F5^p%EuO4sFPrg6-wKsvbPVR~mz+PS@dUez?B<Q)l4(+Yeua5Xr+MQ+Mp+r`zk@ciY?BKYQyveTjM0389P01Dc}%?+>tCFqYY57kz^W=I$SHF!19<N2ho8gyX!yb0<1#VPOqNX*$>9IfsL8x}d^!PxAnA_P{whb3v^aix<-2TP6UJgMu_Ms^fPPTqpcWzZeMHDZn%T=-UQ(PMy4$i=4dZAN&J5b;pmM(5st8-`|E;ciHmlkS}5SNX7#_FiBs~`u+nq5V-f?F}T3}0|R01A96_a$;jQb)0^t#V3ars&9B91X?ih6xU4nmgIye-qOPi{XM{Y%=v>6P;?Qc-S&VMlF8GdfbLq_lbg!Zd2j=9s6gJS7xmhTA6(0k#KTGBq4obYKyzGH{a{NLV0vwi6em7jIkb9lIP6No4s|7;74$|qvinZaB{B-zQfA(qa@~gOqugmPXvY!YxW}bXD7+VIUL?0>Lg0JmUvN?Gw?{aCPl0$g}8yM+t0=^2Jhx#~x4*)Pnh6;Dkk(l<&UMt6^#=(;D<|pjc<43v6>_v}pT1bvDfypWl^k7Da?99>U?o&XP^YFaGWl?5MG+m$Fse3+s%{Zu(Ce_NRGtU++J@5z<Z1xQE6#d{?AdtXm!w_8P^3BNz0xZ;qs~kS=x6WKB;;0+$00Bg!D+th85hG;yB_abQujM|sILC3_GS_#!Z|$t0!7R=nkizK9FOKXW8;!NBzVCQnp5Nb|pM_V;5nAfaJvBs|5%!)Ipp6v>;r1AW*dF%q{K&3n;d!kMc=mC7$A775dk)ndPhI+pJc1hH#gh|sesweq+c=hTkh$hP#&6F3$=*pygYC!}T;TV`M*<J38_GGPxfw0cIbMLR$AwUg9{CiuJQ=0J`d)Da$0<U<o<9Fb+BB~^S^CX_fTCSq=c`+TX12(<GOe&#J3wDC@qMPR4Tp;dhctjv->2iyeCpM0GHh)t5xAOfev*<iG1L`jkkM0H`@F}nuHYq1U!yvtgAadmbN`<gdt~4~EPH!Vfd;`9e0Ud%oX)XLXLNJ?j=>q--&L=T2nPZ{B2Ec9{Mr>O^>F^^4VNN>B&!Cj;prB-Oy-W5Th8^vfbWn=%SdvEcmMw8=Hr&B3mpD{gLL!EG+rH=bujCT<_k5K0&vzZ2C@2VJce|5sv)~2ZaaTP1|2RaFo@OIak-5w2{ZBu{^XnHnU)(&LAo+4U<)t{1J2g0P)$c~4>S4ZCcU!3W=iJuWa6<YgZ&uYq$f1d{Xny`CRI-WbqdGnK_)G&(j*(jY~SQ5CG(r$VQ9e20KnUqvz5*5a~Q*k;cDZgQW!i9iB_gyU^Ei|<tb(+sRCf+mL#(v3R(-)^*pG>CIifg8;`y@SynffgqY39%#1vos=Lbb)kr<6U;%vHW0_M|#D=aksirza^t1UyHUajNUhIr>CW{WpW}1Dck<i;ef&)J6Ba(;HRbM4c=<8?D!u7upjY;&jRt5qAQ2Jq%GcW*)L@gg{qa{zNzM%+eC>ntY&c;g=^u}sI?7^X#<qJv*b?bVtUU8Ix2~Un@rN>bd%y_QykXKa7V`KXIS9r?(d|fvFtICF~=QMF{H5T-u7pW+Ez~?}375FXnd~3lTVjc@|q&UY57Gc0C@6~+MGv69+{YZ+d8Cd~f5WseHiioU1bkI^4v01qdMl3H+PbbZw)ru<!F|^Mhf(4!j7NC%)jK%}%oY4+hCpI(>x=O|bw$IUd6)V8%&BkzwBt#f%xWVsa#ac{jf<PAd`+x*j<le9nl;T@9lD8@Aapjc4G+&Dq;J0X)Pxo3NCoFCL`PnnoBo4CF*ER{5!;e*humAY&{U09_VQ|3BU2ln1+EA76@xK`8@y9D0C%~I^Lxmf+;p+q6hUTKs;#Bj)YY=xcXF77iP<2L{gey2V05Dt`f{2Kz8%j2SG0ULW%27|Y=8-C4r4%CJQfg~PsR5dxtuh8=l9@TQpfVpONYK5NCq`IpjY@C%O4Kb#X|32lBB90O-E`}E?+E--W#n6&A!6#oD0^&*STA6%sM{`bjv{EG(nRRZ%0lm@{R8#@FX>~ot+qzdsypszH;kDVO=oUpQl7k=BK5bwdH#F4^jRj#6o+Z;ZQ!3%KMq=VT!F6AmAye(xte$GVBX_W`Fw(ofkad;=<gZM0@eLC)i1l}6jA}K9UhS5j45tcVC`vPlrSIC`J!t(pw+}SDHpCBC~2PoE;bMcd}SQ&{quwyNY@4cn0dJLD2i0;y0PT$@hcp_l(I4t{(fD7iF$#+<|`Kn@CrSM%c#oD9M`Zwj8)?|wK4>r02aUPhvao(l$fmn(1v_7#Xr|v@vBj(S>RKr!mOntS*P&gY9ZM#nLglT@mE26s<AYjhk58;)-R$3(8J+f!9v1TgRG)DTORoiq<aT$tp{ie@L#0K|I(c>u;K+{f>1k-7J9{e>r7^-l+%uUcaRyhaX6q0jn<~T74%t!O9Jv{h`@|u)e-sz>Z%p=+EwxVs!HzE713j18B=a6G7<7ac4-D1_=ZX!%Obe8GTI~{k^Y6fz1jfD(j+)I9C)`xDo)h`95@++`66bV;ER8v*wj><OMN74JtKB5mJ$W01=wPQbJLaQrJ;M=$heIIPZAsRM5_Rzu`5TMZvX3i_xb|-G3@iIWqs-a(8akXMR7be005J7HB0nbHI{p5X?OAR>;M4T%`{;2KSbxG8U4G2`rMy;U6C7%?kGj7@^I}+u_D?s%F(F=L!`3!hI2p5uy5`vD$+H!2XeOJC9kP-#PCW75x%&9Qy<%_XiW$|fcH@!AmJt3gfeF<z6;m0e5^5k!8T_J{sDIky&a$|DuX3bB-|J}%f?E-jAq)@y&vV$3}r3hjK>-hVCVqF;Yzgh1(*#z9OHAl(%2N*`zRi_`DTdp6*BBdeoTy=(BE}jL{j|m3;>>Xue&#+O-xrGnKYnxAC06$?r&HHy}6}P2qcyyyxqQ0&4ebvKT^VEF=|924YEedc!{iq$83&R<3q>AF%4+|h9P~OA_;+%fTQIehqPfY#iSR|SkP|%a{q_7Hz++vq*#vA1FXQMk%iZ&WLH|;2|lkBLxLA>u3lJz7<puKn2m}L*~4-qbcoc%r=#O7Gfkd-r*~oDM4epKYUsrVuq9-yQA(m>v@^gS=jT0;4ii1Fqd^-WOkNU{zn=KYiI}9%tRujOU;7Vv$l--+rHe7i!yHExU|C30yMVI`cElQ2dUj=cJ~9i=!6cEFk5ldYJEbOjRySB3$Q7=y!nzFr{GEN4WK1ZhPg}B2KaCqKLPF$pu(hZBS+lJ!<|$C8auodaw51{%7pDqx1QfEPc9&N^RSKLzcuNg8o(ZtI7?^g}WHyv^wTV`7e5Xf1#7$3JYTD~(U?(pFE5z<0C{@{qMUSBr=S!4~gASZ$<<b0s5C=|vb6Z+<eu-JAkI@{n<7q&+NE#SXpoco?jrg!wy-VY7vFQ?`_y9@=Mml(k`P3BXF5`8dO%c0*P~aBEJSK+|J$bWkTV<<K0_Mn~tmtG)7u4g((I*!HhE%-bkX}a$)<uksfR-bq-TExFfM#?l?SY$LKIEYt6kdOw0tv~gUqq6q;sX1fu*sze9l9}Uo%ZO=?&$z^ssui;bnSNqK8S8P&kROa5uR7JS8k~T?rH3$rdcXWsNjY|a?$2q$foKHX{O8yuoWcYg#E~xd{HCFD9kYB5_h`7bv3|Zkc^=R*IS>ZnoCq`n!2>fKPVd;q{^pv|LOk?@!wHuAG#aX!CCb_I7R1i%&&&E0YEJH<CegIt4Pe*SRo@SE(8l16Xf)@MwJy*%XzV{%B1W33IjR>d_))<53!?q;XUI~gSjoWl=Ai)c=4)CK$`9@s+e2C2Q1q?;s{(d*G+_8VA%ZP$<yh*oI_$RDwM_%EgD?ftgC{Fq9he6G_z0(a5zI}n5T8RiWs!wOO%pO9Mt4yj(}^>I<o@?+dlr3&C1w{JE}<ot-zc0N@jbbp&Uh5Vp29@E9A5$=3Bj-WaGErfB(C))u=dOUaF_qn%1nq!G9U)^U2UtS$M+O*7=ZUm2L^ZIbENT+0lb=C`PlBp7?sB$4|Jzv&s$Z|L7_J;T@H*7m~h!g~=p$YlrehQb)5gm8ohh0#J(R1ViJU+`Z$}#IZMGdRZ8)61V7li)Z(A;ML$P?k$mH5n0c0$INvX>P;_($LypuAfW+oKUR{-!7h$VVrtu3weE}*i5$`s&th%(afG8)0r^Dm-oo&jFt0_L=M@c2PLGq+Rxuz?&*dA<3{K4OoPpW$pxC-iDBWs}^Ql5YdvJK&n1>0YluD<L?Z;2DaUA5JouC`1|D=9u*(sn_qD8rE&Lo0^3~!v%nVJ5L@C1lGGo`+VgZHW@x<Mo&6Hh?pDo7-U!#0`sr8}NRpXyt>MPsbHICgIm(o;Eh?v9u`E?fqak`(gBHw6n|)0xqWSzQtU7*4;26qfEM@Yx)0Sl<<zmPC*sxnxO8lR5TaXjnG;k4#7hPuzK=j16oIdI#5SchE6h_wg!yF;i`kkh_gs3az-{P(p+JYwFjrdJs{w(8=(rKp{pts%9P^*JA($o9T%Z?c9b2o{lm}ejMW9u16~=8p=n4mF*0}XeO_T`O;F`rpydWP=^7adWx-a#VyH4gVHZybs6=***j|Gn&_b#yVGfKMJu0ORkBsTR^P^l*#6H&&uaSx_#)DMO@U1XkS-pYNh83V8cvzu$QCO<=h2<X=N3`dD1#(WEnR5#LX#*)9VuOLL|;O}BqE9M#@|vg@}_EGsSaXl$ciE$a=nZP<1REKzOX>NnSRmasGYfMYF+Peaj)fZS@i}G16*3Awad$q`J%czny^}-Ax={&FYannj8~iJxgc>fOT?8q$1f!LAd;P|F0349NoWfUV>_Me<EZzz`_=qSo-Jk~>;cFE1tC_|vtRTDHhn_E^m+WlDI{=}v;;i;8|qawa1>j{CDIsnly7$2R&|XH6M>+B6^WRHt_PGIc4hwHJ<`$=w^a$>zWZM4TB^93{=Qqg0OHV8x_m&v{VW!hN809YF-A1Y!v+lj>Lhks?hay1#HEl`0Yq(dK_!pxeIIu)nCl%S#+*|1Qs|@;<1%KCr@uyp&WTq}ogrTqhchwr9AqZk;*<HC@<KcT&Ya}TvI%v?WmAl4$x0IDo<P1EYxO7>B_v8heVLK53v5?DZtCER15I+F(u~sJRiHx;=gR1xSmv?|8_~BOJOf5q6$|45iB=|hH7lamym-MLNel%DVQVu}ge(%9Ql{dL#MA;5&6HXU$Oj(U!z_y^cvKRp9Qwej^H!w-Mh1{f#+<w*^lv;Ii}jCc5DPI+i}VE+lgVB%*-=2#08acK+_+r#9V+HASBr%iew$lPrOm1fRx`~KU@U}bGOTc=2p5g)x5Dua2^N8X46L+yV!zNV6ZwiP1r?nypfRH2bVQnq`fiXb;<`v{?IOXp6QWV9DOAk*5Xb^o83bDbAD++F;k`yh1o=u}(;PrVz*1qII%ReSE-R&HWlbLz)*abGhpLFzmOO8=wi!qJ_&cy*jAQJgpGJp>3bh+~%Ul~W7I*Wx=Nx(YI#k}9w8l(NlIaid6=s!svKbJ#Rn`M^J=O26h%CJv)|PmPvq6~HO(7g`88<2$Z8Qst!9P1$ytp}Kj9}l|nOGm(WvoxD*o?;cg7#i5X>U}J7J3Qjj&Sv=n4LmxS$Q(>R;|;&Rs#bILme_!esr)t=d)RsUiPa}V?B_lD}aKr^AgoWt!D>(f0GT<BxI4Jg>7h3T0z&v0op3aCU_m-Ar%m!c#m-?VpBon$Zl|wH~g}dIm^XdJcjww)<8<g5+Y^U&r;xLV>m}j_e5!(0~b9K4rbotA}|p^LZ)Es2{;$XN{3Ho;@J6MFBd~YrzgZW(wT<UC~azmOa02EfDl1CO#?6@Tp?Gzpv#G9D5a-#pbcY{7(5hOV>*}{(U3tY^xE`0=e5YM@i1c%uyJMM`|VtcE)}sB3*`b`7R&SARC++g5LOXRllJZ?+7lL2v~Q_@N43%_;_HHB9Bo<aW|Nc<YU>lDHX>fVF0~2CD$WD(286vVP%wKKuyO*{^%&tvjUc7D_>}Re(STf7IHy%<fM{-1JXWAyG<YkKJEj_Zk7j9tjubkjTN(L99OZ>~NwGkb>Khj1jTLt-152pPN1reeuMFMS6Q0SU0x17!xX8B{C`*5W%e$hPxn59!=3rg)Kq@z>TO`F~hs9{R)gj6nzWZe^I66*XRG|z5dfE}gQZvzNPGjo8M2uNBEV7z&s@@q)O9+h4@U0KgCKI<+*Dy(wrR7c@@R*9c!e(Khnk$QNX8*=0ky*__mep*vUJr#`UVQt!oQ)QZjhW<7wN=>=nIg>9BCd|p^wZu8l-x3~D1~AYKu&`Y51tZpG4jAMrLX2MbLZ%bE$l60Yi5eHcB><k&Dy6WTa~r)l{YE-QF*ls0|;<AfK8F-EwCABlx)`tcCvoLZWj#EaSU;$jmA0hv}D6ADKAs0HDis)Jz<11zUD1!1BFW^pmtop-Q$?zC|`h!qJ4Dz@It$i1Xo5aKTS7Kyo-{BRF~&7bxSw8w;Lw`a5O-{`SRrelno9@z(9i<kiQ3Q>4k<En4#;p&W#JSyvcL#^;MucVh+f~K|ayWx^efZgQ-<UH$l}in0H$649PJLXz3Xh>99m;giLG>{0^?oUcunYe{+oH<y}FHTGFn#4Y3kb(Cc>@=vwJBrWHL&Z5{n9KhM`ql0_(t75duFp|hb?R*a=fysquA(hNXSB(U{L1NYYQy3%7zFxvzI&ilUjC(xrx(5l*Cw*)&~{Xb#*)Ww1DEocY`O`s`PH>lO{h}+vz0T{CDi&8<n9TIYF8`);gv@DUS&p;FE(X$q++l=q1WOHiR0;PGTr7{y7kfL_xG>nL-FkiM20#!T8w^daG$%_zqt!A$hf>T*)i?r=UaI=QAUaGc}&w`3dS~aCvt8(lnwS#HE1J97Hivx?6x<<Y-Q7p^sWKbgaG(d`~{1tHV*)c$d`{*|Srae5O@;M=S*Aa3ykq%2%or!68kIu2VootbP{C-3ND?w*~DSAn&v6?_s?w$}SwSDGCh!N(zC^skK8i64M)7H1gRVH<15n(ar{NZ~Xe=@yu1jsPyYofY2FyhlZ+-}@bml-l}2;l{Fg|cpx_ExX)069Yf0i-96lrVyTowg*k^Tb8IzphHHvN=gT#;(apXBBu?aWOo(l+6?l14bc5DoIuY<fI*3I1YG`nBA>4qi4=qYa>WaWK=;Tt`u{Py|`k<a9oIq9B=;DRH~b>xMDA<#~mR1iY=yZRaJ$~3#)BxD9s>t<;=zgeQUFbzD1NxeF~P3(v%u;=efG7T4AT8ZxQ3H*#*eubUB(zTcru*(R>N`WwSt-<J7GH)X|PSX4akUVe&h6g7Z{hT08q7j#KtOVx%a$*^#6`QQ8Bmr{cZLBt>~qV>8giq(V=06ee6v#bV8pV=LTu&YT~S9bN8!mV_~MhbfwE?{)G4TPkaTE=qRPK~I~yjID_&Wc7Kg3C6;pu(V2F_=P;~mDooM$_rD)+`0Hyy<s%uR;j2Z<g|OCyWdIEs|MYJN`i0It=bcI=5<7;lt>m9S1*J@Jqk6J0|N@<4=`Sg-;#&^Bk~#o7IIrK2th{oG9MwH&HB`*+;fNFli?Vtbn58)gd;K_UNd{f13bENjQtKNSg<&td4Z3pGsU?kva|fN0_tS~SF!#tN+s<N0#k7GgqA`nRl!KUf$}}h5`>VB<W%h#K|beANygx1>io;s1ko2H`9|WDg4IcpBCmR^lcXL8N9B9?f|UBwXGyrz%xk6a7!0mZT0%m5CtoOmI6uV4Z)fv!7;8#~MwA(1Ee@><AUAYH-yBloX*X4;bzI`|5#?ZiUWL|TCRyrsoo-Z=xuLPJ2z)P|4beqld9J)gY4KE|d5EwL)9_SNB`f|4`a^^<Sns$Hw}Xz+l(FILD^m`MdqJAMq#%*D%1V&bB=e1;$WU1#D(mE2WeWc_^2vR@c|RM*!%-OKP~8JF$NZ|_U5??IyxW|kqH{h5oov8HYg%=^^o*f9NV$Q>#ZJe*q@%ejP^ugYkPT;uPg?IL;6H9fV8)bGb(h9xw1p(yXv|$~@b+F60Sehh4U4`P&n6D&EdY-^dK)c_l*9y4v{1=QI3<#{(pFpzf$XJV6zjwmA+JnZ45?{D7-Rp3Ak8Ip#?6atdP`6G%b%QCuPQP2H(9sG`;L80wW#PTbvwJU8=JJkR=b)vLxWS-j{A62f#Pi9mlseDkAkqkBeBcVK+Qpe4`wv|D&^u?w%IP^#=ozTv(CjlLV^b(B9BJ_Olie5YN1(Ru8i$tPCU_ObI7uarrJIcSznN_nxJE$%0X8of%8GmdU9)rjEz%-sv;3iI3G$@0uB^7mE4AIEc5F$Udg<cTpDC~b)MhnfK#<t03Q<k=|r7Ru{oYhPT`YIbDBARk-KkU4>lrr+@%V9;jIg_%T{i+wI3R4z2LOSj;@<Zqb29Zf+!Lz-wFxyo!yW&0hS;Yj%g-&)o(*l*5s9a)iLVSmd}nV6OJkd)edxImGe-&)RmF*k!UsOe$VBoKPJNDQ4+xcsywT(k@lk%uV-?2U}J(@;HSc{%3X=cn7~K+jzM?VoHo&^cInjYigpFeSwMg>I>ch_n5P<QdwmS{W3?)VR-JUG)Ph-u!u~4CI+!{1#TYrjUYw7Al~ln;>Grsp^x+U{`JVmVV{2vp*1A%&2l2Iyzu&{`02_L-sFts)gT?V0wpDs%OT)CSc4glF*=qO7idMikaGTgau|U=CEM4wYt@mKnswDEXC&A|DC)s$i9c-#uqb9CWXy{i5MZ`(<z~wE{n;5s-+flrPsYL_a3`AiX!9O9Ipt3WY9lVRWE)F~_kkMg{gWCN-L5yKw`i$<9y1hdKcV7g`=sCYrYIcJeIaUcR0=hh1+q@aZla<QmvyXQk4oe&!3wFJgGh%1rqHie6hX@uA&iG+~9uxY<rJ9*rxvpK6UJaWZE$ybRd<*EF7qm|`m}TEVpnn9Rr6gotq&ds5q8zDPNPV0H)iDnOof8`zmAz&oSMAxIpI6mI?;zr!HK&w8W@&jD7aKDcREBy$W`tlROM0OiXj4%q$)LndfPlgH5i1njk**$A_3SAPepURu(4&ae4ML|ep+1LNLqqfQMeKRF3hURZG(2UDK7E<tU}>qfN&Z?`i<qHSbT)N4Vssdz+tiY5mwb{ijpUY(C5X{iBy!G~v}D0CU{%9Okrgl$3=#FjsP2!dH``OK*iPGA`j~EiL8ei~d-QNp`s~}5mAuvEhRnX<#y9CM?xZzAyrL!L3$RojUO^$cXk%H5SA<R|Rh4AP_B(e)0gz`xPe8#yyu^9x46FRsqDGaF1*@GVR-dWNCbot+>e(g7*%mG^Gr{}Ys`f)ESJFZh?MG?xIe)zXfEv?^(yw{Z<#CQ`&YaRV$~xhBm&v?)9a<UhCerpt+$nO3V?<bp;!}8f7}%Duu-sch1oez_LaxoTb@&hjAJK|TdbjZ3KZFj;#G;tSP)cAj<S0>ZID>t)E~sbV4XDj3aN9w;Do(970;L455=qRY8F)LwpOO-qj&6X_RCz%jj$6->IVE8>!~p>$)F^A7-{J8nCVO+roE%GzdP4YVq^$Wm{Y#f3b=B$SbPi#2OUkw+Yd-rY@p}Gnd-L}5n@=-%Gx^OT9rTJuxN;>6AsOHJRNw1NV(Hkbi4wW!5Z9T*E`AkbuR_ZfCyL3%lLj>w1keyoQx&=bYv;?Ro^iXZ=&pl>G8(xsqINLR@tj3)YS4Db)qZO-Gr9;cigRC96tQ((i_u3CR<RLlwwkmiix9Ny;i+ziXD!#x6oox7cs!Ab9>!TMK`5i9!fib=Y99c1iZE$Sb*}GWv=3LHW^Y?L99manMFsmAO<Vc&V@_tyHw(3;aUmr>Ao`sd7DlK~Mft_RSB2alCcSj7h6%|*?kUl!E@{GPk1Jjh@gnL(RYiL6k*cJkv?G>1Jv2;{zCkK3WO*dA%Pn#y4sTuW5$MT|3ET|dql#x=mDa4+>aqjWDs*Lvv$`E2t%+Z0+k(6lY&KS|%QC8R<A;4!n(P|L_-MNv<p8i6V5rSn3YG|Jp?BcJ+4Zu*T=_e})XLR{n}V2Eq1bQ&F}Fp#N_B-TNvJZa=4ZO=wT8Z2mrc`ItRK3t=&&WEM%YGAQH;n6C5!9fKwRC%fK66W-eygvFFNLB1Wr8(eiFi4K*n;ONTX^I>=uYgZcR1MD%*(p=46k6t(hTmPON{10Nvnp=tGq;1k#p9O-36|!ISb!CSd7^(op^OG|zv=+DmXcgB@Tw@s|8^0*x(;-wS7hI=Ic+L{9_<L*z1)Uv+OKQn=_Bn%V_(B>4KWuQqK;pOel9D3Iud2ui9l;OxBeWMCjn&$Q7{fG*vzm$RvgG7~`dkdkXP50Ty$ix_9ZvZ9-x7DJ2T!>!39TN|K;3uDlPnqW_O9qsM9O+~B3*cXE};kNRsBugH)*_w|~@e0&KBs8}V)`{ip_3hmJi`eg!s0e#rakj6;(G)Q9Dt0WCdtCK5!Tu}(+wiG(EXB{Ew75d2NbMX;yZmQSWGspt+>mjD${*KfD}VF_i7NGvWb+ieF*>D?O5nFp38{6A&RqD|=Owj7XqVSOQkqXeTtaX=D&k%&FpEEwMg|UKM-OuZKa3%r6;2`57@&lucGtpw5wO!3s?=zqBX|e3y!;9_JU-?{j*H?Eei<qXI;e#w&RfAPc{3E&38H}3C97BRy%T&?^kULEUxw8W?$6?9JxT}4&MI({RN&|R8enwk)1nkpaVH6?*+xydCQPDE6G}v2?36@Fo0qjfRCldpGW<X{+%Y2fth6#@u&gl5OuhR|{Y!ix)n1>>@(B8x>r7dqUeIb$TFe3^Q<<cas+wD&oEwjADV%p#rAB#2@0?hs{8*_!^2{}25R^*psBpKDX{Ssgb~3+Y8c<j^pQ4`bKAdr!4B>U9{6^_@;4J6c`qkMnn!UKs0e&*tY7fF-HpMbQ#PBn;sEq!2r!#=)PrCI8D<-r=C9)PI;qtFVa54?=OZxPC=ji~Jv59uM(CT5OmLN=R!qf5wf*LxjK}l{wdeylnJQl89+-URDMhn;t=hg(nIG5Asr0YPV4FAnnm9{BvN~tfpu1*umb+==vp?wG?{acCKiv|h8!DkziN_SENQpIQ;`b%2W+m|)Vd~}VzB+~cdwTi|LOv9H5))1Ev8##BkO(UnSI+~l_k3j~AnM=_E3){8qdZ${%(Vp<N=CMo$+pPWpbLluGAb{qqFSB>>^h4leF_qE>-{P{*yaCZNE!Hwa>G^Azp+*Q<MD5F7Cbd95$RmZ+`azgYGJ-aBf?h+s9VM}r9oZ+`eUvTfz&qfwyqL}|)lIMhDe9liKQieCMeEw-jv7~2X4;39Y9GhY5S?NJgZe_{8&W-sYUVQrj?i)^M5_I`HDV>#h<^Wi{S^w_#ObpmhC&WA)8dhpBz1&{?e#jjZ{pFg4&4AYt*P<jc~26i?_2wtI{s@^hPN}FH(^ZW*I1G^^A*aSYLED4vi9s*1)x`*aeA^}tW85L1%|Fbq>XJhc&;vNA+;&QzpT<0S%$}!(n<Mo{<i0a^YqZOVvbWK)f#d@bsVI!_G0UxIHC}nx0uCXNaB$tU3oSL2d&_h3N%`t-KVzES<$O2-Z*pLN?L_Zcb&&lC*;Gn%)!kg<z`dgK_G;SvxUb)6)pVk6LY`qd7NPxxgDK&CHX1>QzIR>qOQLR(^gVvHiPwEIIXwBJh0R@=<PJaR}7yePRBE?oEyToIB>8yag;F~U#tvCqqob#m}?`9Kd?=4tiTJCx?q+|!;3#K_2l4Wq*DxHw#%2Oz@qSCLw;b^lVM|5554j3{ttlLl5q')))
agent = make_agent({0: _DISTILLED_ROUTE}, opponent_plan=_OPPONENT_PLAN, **_DISTILLED_SETTINGS)
submission_agent = agent


In [ ]:
import ast, hashlib, pathlib

# The agent was submitted with CRLF line endings. The cell above stores LF; restore the exact submitted bytes.
EXPECTED_SHA256 = "4b4a4c4b63957b8c095f789a045505c7a238999d315d7f518237e5a7b9f34db6"
src = pathlib.Path("main.py").read_bytes()
exact = src.replace(b"\r\n", b"\n").replace(b"\n", b"\r\n")
digest = hashlib.sha256(exact).hexdigest()
assert digest == EXPECTED_SHA256, digest
pathlib.Path("main.py").write_bytes(exact)

namespace = {"__name__": "agent_module"}
exec(compile(exact.decode("utf-8"), "main.py", "exec"), namespace)
# Kaggle loads the last top-level callable; `agent` is built by make_agent() and `submission_agent` is its alias.
assert callable(namespace["agent"]) and namespace["submission_agent"] is namespace["agent"]
tree = ast.parse(exact.decode("utf-8"))
last_assigned = [t.id for n in tree.body if isinstance(n, ast.Assign) for t in n.targets if isinstance(t, ast.Name)][-1]
print("main.py sha256:", digest)
print("bytes:", len(exact), "| last top-level assignment:", last_assigned)


In [ ]:
%%writefile LICENSE

                                 Apache License
                           Version 2.0, January 2004
                        http://www.apache.org/licenses/

   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION

   1. Definitions.

      "License" shall mean the terms and conditions for use, reproduction,
      and distribution as defined by Sections 1 through 9 of this document.

      "Licensor" shall mean the copyright owner or entity authorized by
      the copyright owner that is granting the License.

      "Legal Entity" shall mean the union of the acting entity and all
      other entities that control, are controlled by, or are under common
      control with that entity. For the purposes of this definition,
      "control" means (i) the power, direct or indirect, to cause the
      direction or management of such entity, whether by contract or
      otherwise, or (ii) ownership of fifty percent (50%) or more of the
      outstanding shares, or (iii) beneficial ownership of such entity.

      "You" (or "Your") shall mean an individual or Legal Entity
      exercising permissions granted by this License.

      "Source" form shall mean the preferred form for making modifications,
      including but not limited to software source code, documentation
      source, and configuration files.

      "Object" form shall mean any form resulting from mechanical
      transformation or translation of a Source form, including but
      not limited to compiled object code, generated documentation,
      and conversions to other media types.

      "Work" shall mean the work of authorship, whether in Source or
      Object form, made available under the License, as indicated by a
      copyright notice that is included in or attached to the work
      (an example is provided in the Appendix below).

      "Derivative Works" shall mean any work, whether in Source or Object
      form, that is based on (or derived from) the Work and for which the
      editorial revisions, annotations, elaborations, or other modifications
      represent, as a whole, an original work of authorship. For the purposes
      of this License, Derivative Works shall not include works that remain
      separable from, or merely link (or bind by name) to the interfaces of,
      the Work and Derivative Works thereof.

      "Contribution" shall mean any work of authorship, including
      the original version of the Work and any modifications or additions
      to that Work or Derivative Works thereof, that is intentionally
      submitted to Licensor for inclusion in the Work by the copyright owner
      or by an individual or Legal Entity authorized to submit on behalf of
      the copyright owner. For the purposes of this definition, "submitted"
      means any form of electronic, verbal, or written communication sent
      to the Licensor or its representatives, including but not limited to
      communication on electronic mailing lists, source code control systems,
      and issue tracking systems that are managed by, or on behalf of, the
      Licensor for the purpose of discussing and improving the Work, but
      excluding communication that is conspicuously marked or otherwise
      designated in writing by the copyright owner as "Not a Contribution."

      "Contributor" shall mean Licensor and any individual or Legal Entity
      on behalf of whom a Contribution has been received by Licensor and
      subsequently incorporated within the Work.

   2. Grant of Copyright License. Subject to the terms and conditions of
      this License, each Contributor hereby grants to You a perpetual,
      worldwide, non-exclusive, no-charge, royalty-free, irrevocable
      copyright license to reproduce, prepare Derivative Works of,
      publicly display, publicly perform, sublicense, and distribute the
      Work and such Derivative Works in Source or Object form.

   3. Grant of Patent License. Subject to the terms and conditions of
      this License, each Contributor hereby grants to You a perpetual,
      worldwide, non-exclusive, no-charge, royalty-free, irrevocable
      (except as stated in this section) patent license to make, have made,
      use, offer to sell, sell, import, and otherwise transfer the Work,
      where such license applies only to those patent claims licensable
      by such Contributor that are necessarily infringed by their
      Contribution(s) alone or by combination of their Contribution(s)
      with the Work to which such Contribution(s) was submitted. If You
      institute patent litigation against any entity (including a
      cross-claim or counterclaim in a lawsuit) alleging that the Work
      or a Contribution incorporated within the Work constitutes direct
      or contributory patent infringement, then any patent licenses
      granted to You under this License for that Work shall terminate
      as of the date such litigation is filed.

   4. Redistribution. You may reproduce and distribute copies of the
      Work or Derivative Works thereof in any medium, with or without
      modifications, and in Source or Object form, provided that You
      meet the following conditions:

      (a) You must give any other recipients of the Work or
          Derivative Works a copy of this License; and

      (b) You must cause any modified files to carry prominent notices
          stating that You changed the files; and

      (c) You must retain, in the Source form of any Derivative Works
          that You distribute, all copyright, patent, trademark, and
          attribution notices from the Source form of the Work,
          excluding those notices that do not pertain to any part of
          the Derivative Works; and

      (d) If the Work includes a "NOTICE" text file as part of its
          distribution, then any Derivative Works that You distribute must
          include a readable copy of the attribution notices contained
          within such NOTICE file, excluding those notices that do not
          pertain to any part of the Derivative Works, in at least one
          of the following places: within a NOTICE text file distributed
          as part of the Derivative Works; within the Source form or
          documentation, if provided along with the Derivative Works; or,
          within a display generated by the Derivative Works, if and
          wherever such third-party notices normally appear. The contents
          of the NOTICE file are for informational purposes only and
          do not modify the License. You may add Your own attribution
          notices within Derivative Works that You distribute, alongside
          or as an addendum to the NOTICE text from the Work, provided
          that such additional attribution notices cannot be construed
          as modifying the License.

      You may add Your own copyright statement to Your modifications and
      may provide additional or different license terms and conditions
      for use, reproduction, or distribution of Your modifications, or
      for any such Derivative Works as a whole, provided Your use,
      reproduction, and distribution of the Work otherwise complies with
      the conditions stated in this License.

   5. Submission of Contributions. Unless You explicitly state otherwise,
      any Contribution intentionally submitted for inclusion in the Work
      by You to the Licensor shall be under the terms and conditions of
      this License, without any additional terms or conditions.
      Notwithstanding the above, nothing herein shall supersede or modify
      the terms of any separate license agreement you may have executed
      with Licensor regarding such Contributions.

   6. Trademarks. This License does not grant permission to use the trade
      names, trademarks, service marks, or product names of the Licensor,
      except as required for reasonable and customary use in describing the
      origin of the Work and reproducing the content of the NOTICE file.

   7. Disclaimer of Warranty. Unless required by applicable law or
      agreed to in writing, Licensor provides the Work (and each
      Contributor provides its Contributions) on an "AS IS" BASIS,
      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or
      implied, including, without limitation, any warranties or conditions
      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A
      PARTICULAR PURPOSE. You are solely responsible for determining the
      appropriateness of using or redistributing the Work and assume any
      risks associated with Your exercise of permissions under this License.

   8. Limitation of Liability. In no event and under no legal theory,
      whether in tort (including negligence), contract, or otherwise,
      unless required by applicable law (such as deliberate and grossly
      negligent acts) or agreed to in writing, shall any Contributor be
      liable to You for damages, including any direct, indirect, special,
      incidental, or consequential damages of any character arising as a
      result of this License or out of the use or inability to use the
      Work (including but not limited to damages for loss of goodwill,
      work stoppage, computer failure or malfunction, or any and all
      other commercial damages or losses), even if such Contributor
      has been advised of the possibility of such damages.

   9. Accepting Warranty or Additional Liability. While redistributing
      the Work or Derivative Works thereof, You may choose to offer,
      and charge a fee for, acceptance of support, warranty, indemnity,
      or other liability obligations and/or rights consistent with this
      License. However, in accepting such obligations, You may act only
      on Your own behalf and on Your sole responsibility, not on behalf
      of any other Contributor, and only if You agree to indemnify,
      defend, and hold each Contributor harmless for any liability
      incurred by, or claims asserted against, such Contributor by reason
      of your accepting any such warranty or additional liability.

   END OF TERMS AND CONDITIONS

   APPENDIX: How to apply the Apache License to your work.

      To apply the Apache License to your work, attach the following
      boilerplate notice, with the fields enclosed by brackets "[]"
      replaced with your own identifying information. (Don't include
      the brackets!)  The text should be enclosed in the appropriate
      comment syntax for the file format. We also recommend that a
      file or class name and description of purpose be included on the
      same "printed page" as the copyright notice for easier
      identification within third-party archives.

   Copyright [yyyy] [name of copyright owner]

   Licensed under the Apache License, Version 2.0 (the "License");
   you may not use this file except in compliance with the License.
   You may obtain a copy of the License at

       http://www.apache.org/licenses/LICENSE-2.0

   Unless required by applicable law or agreed to in writing, software
   distributed under the License is distributed on an "AS IS" BASIS,
   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
   See the License for the specific language governing permissions and
   limitations under the License.


In [ ]:
%%writefile NOTICE
Kaggriculture top-100 public replay distilled agent, 2026-09-30.
The route-replay chassis is derived from Apache-2.0 public code.
Original attribution and license text are retained in main.py and LICENSE.
Selected action routes derive from completed public Kaggle replays;
no competitor source code, opponent private observation or full replay is included.
This notebook rebuilds the archive that was submitted as Kaggle submission 56720329.


In [ ]:
import hashlib, tarfile

with tarfile.open("submission.tar.gz", "w:gz") as tar:
    for name in ("main.py", "LICENSE", "NOTICE"):
        tar.add(name)
with tarfile.open("submission.tar.gz") as tar:
    print("archive members:", tar.getnames())
print("archive sha256 (gzip headers vary between builds):", hashlib.sha256(open("submission.tar.gz", "rb").read()).hexdigest())
